# Grupo 3: Cadena de Suministro de Ayuda Humanitaria

## Título y objetivo

Este notebook implementa un modelo reproducible de inventarios y distribución de ayuda humanitaria durante las 72 horas posteriores al terremoto en Ciudad UVG. El modelo distingue centros de acopio, zonas, suministros, rutas y vehículos, y utiliza bloques temporales de 6 horas.

## Imports y configuración

In [27]:
from pathlib import Path
from dataclasses import dataclass
from typing import Any
import re
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HORIZONTE_HORAS = 72
PASO_HORAS = 6
N_PASOS = HORIZONTE_HORAS // PASO_HORAS
N_RUNS = 100
SEMILLA_BASE = 2026

RUTA_EXCEL = Path("datos/Grupo3_CadenaSuministro.xlsx")
HOJA_DATOS = "Datos_Grupo3"

assert N_PASOS == 12
assert N_RUNS >= 30

# ------------------------------------------------------------
# Configuración central del modelo
# ------------------------------------------------------------

ZONAS_ESPERADAS = [
    "Z1",
    "Z2",
    "Z3",
    "Z4",
    "Z5",
]

SUMINISTROS_ESPERADOS = [
    "agua",
    "alimentos",
    "medicamentos",
]

MAPEO_NOMBRES_CENTROS = {
    "Bodega Central": "Bodega Central CONRED",
    "Bodega Central CONRED": "Bodega Central CONRED",
    "Almacén Norte": "Almacén Norte",
    "Depósito Sur": "Depósito Sur",
    "Centro Log. Z5": "Centro Logístico Z5",
    "Centro Logístico Z5": "Centro Logístico Z5",
}

MAPEO_DESTINOS_ZONAS = {
    "Albergue Z2": "Z2",
    "Albergue Z3": "Z3",
    "Albergue Z5-1": "Z5",
    "Albergue Z5-2": "Z5",
    "Albergue Estadio": None,
}

configuracion_modelo = {
    "horizonte_horas": HORIZONTE_HORAS,
    "paso_horas": PASO_HORAS,
    "n_pasos": N_PASOS,
    "zonas_esperadas": ZONAS_ESPERADAS,
    "suministros_esperados": SUMINISTROS_ESPERADOS,

    "mapeo_nombres_centros": MAPEO_NOMBRES_CENTROS,
    "mapeo_destinos_zonas": MAPEO_DESTINOS_ZONAS,

    "factor_distancia_retorno": 2.0,

    "fraccion_velocidad_ruta_danada": 0.40,

    "factor_tiempo_ruta_parcial": None,
    "factor_capacidad_ruta_parcial": None,
    "permitir_rutas_parciales_sin_actualizacion": False,

    "centro_receptor_reposicion": "Bodega Central CONRED",

    "acumular_demanda_no_satisfecha": False,

    "incertidumbre": {
        "cv_demanda": 0.0,
        "cv_tiempo_ruta": 0.0,
        "cv_reposicion": 0.0,
    },
}

assert configuracion_modelo["n_pasos"] == 12
assert 0 < configuracion_modelo[
    "fraccion_velocidad_ruta_danada"
] <= 1
assert configuracion_modelo[
    "factor_distancia_retorno"
] >= 1

print("CONFIGURACIÓN GENERAL: PASA")
print(
    "Bloques:",
    configuracion_modelo["n_pasos"],
)
print(
    "Zonas:",
    configuracion_modelo["zonas_esperadas"],
)
print(
    "Centro provisional de reposición:",
    configuracion_modelo["centro_receptor_reposicion"],
)

RUTA_DEMANDA_GRUPO1 = Path(
    "datos/output_grupo3_demanda_suministros.csv"
)

RUTA_REPORTE_GRUPO4 = Path(
    "datos/Grupo4_Reporte_Infraestructura.docx"
)

CONFIGURACIÓN GENERAL: PASA
Bloques: 12
Zonas: ['Z1', 'Z2', 'Z3', 'Z4', 'Z5']
Centro provisional de reposición: Bodega Central CONRED


## Incorporación del intercambio presencial

### Demanda recibida del Grupo 1

El Grupo 1 entregó una proyección de la población que requiere suministros en cada zona durante los 12 bloques de simulación. El archivo conserva una estructura ancha, con una columna para cada zona. Para utilizarlo en el motor, se transforma a una estructura larga con una fila por combinación de bloque y zona.

Los valores recibidos se interpretan como personas que requieren suministros, no como unidades de agua, alimentos o medicamentos. La demanda de cada suministro se calcula posteriormente mediante las tasas per cápita del Excel oficial del Grupo 3.

In [24]:
def cargar_poblacion_grupo1(
    ruta_csv: str | Path,
    configuracion: dict[str, Any],
) -> pd.DataFrame:

    ruta_csv = Path(ruta_csv)

    if not ruta_csv.exists():
        raise FileNotFoundError(
            "No se encontró el output del Grupo 1 en: "
            f"{ruta_csv.resolve()}"
        )

    datos_grupo1 = pd.read_csv(ruta_csv)

    columnas_esperadas = {
        "bloque",
        "hora_inicio",
        *configuracion["zonas_esperadas"],
    }

    columnas_faltantes = columnas_esperadas.difference(
        datos_grupo1.columns
    )

    if columnas_faltantes:
        raise ValueError(
            "El output del Grupo 1 no contiene las columnas: "
            f"{sorted(columnas_faltantes)}"
        )

    if datos_grupo1.duplicated(
        subset=["bloque"]
    ).any():
        raise ValueError(
            "El output del Grupo 1 contiene bloques duplicados."
        )

    datos_grupo1["bloque"] = pd.to_numeric(
        datos_grupo1["bloque"],
        errors="raise",
    ).astype(int)

    datos_grupo1["hora_inicio"] = pd.to_numeric(
        datos_grupo1["hora_inicio"],
        errors="raise",
    ).astype(int)

    bloques_esperados = set(
        range(configuracion["n_pasos"])
    )

    bloques_observados = set(
        datos_grupo1["bloque"]
    )

    if bloques_observados != bloques_esperados:
        raise ValueError(
            "Los bloques del Grupo 1 no coinciden con 0-11."
        )

    horas_esperadas = (
        datos_grupo1["bloque"]
        * configuracion["paso_horas"]
    )

    if not (
        datos_grupo1["hora_inicio"]
        == horas_esperadas
    ).all():
        raise ValueError(
            "Las horas iniciales del Grupo 1 no coinciden "
            "con bloque × PASO_HORAS."
        )

    poblacion_larga = datos_grupo1.melt(
        id_vars=[
            "bloque",
            "hora_inicio",
        ],
        value_vars=configuracion[
            "zonas_esperadas"
        ],
        var_name="zona",
        value_name="personas",
    )

    poblacion_larga["personas"] = pd.to_numeric(
        poblacion_larga["personas"],
        errors="raise",
    )

    if poblacion_larga[
        ["bloque", "hora_inicio", "zona", "personas"]
    ].isna().any().any():
        raise ValueError(
            "El output del Grupo 1 contiene valores faltantes."
        )

    if (poblacion_larga["personas"] < 0).any():
        raise ValueError(
            "La población recibida no puede ser negativa."
        )

    if len(poblacion_larga) != (
        configuracion["n_pasos"]
        * len(configuracion["zonas_esperadas"])
    ):
        raise ValueError(
            "Se esperaban 60 combinaciones de bloque y zona."
        )

    return (
        poblacion_larga[
            [
                "bloque",
                "hora_inicio",
                "zona",
                "personas",
            ]
        ]
        .sort_values(["bloque", "zona"])
        .reset_index(drop=True)
    )


poblacion_grupo1 = cargar_poblacion_grupo1(
    ruta_csv=RUTA_DEMANDA_GRUPO1,
    configuracion=configuracion_modelo,
)

print("OUTPUT DEL GRUPO 1: PASA")
print("Filas esperadas:", 12 * 5)
print("Filas obtenidas:", len(poblacion_grupo1))
print(
    "Zonas:",
    poblacion_grupo1["zona"].unique().tolist(),
)
print(
    "Bloques:",
    poblacion_grupo1["bloque"].nunique(),
)

display(poblacion_grupo1.head(10))

OUTPUT DEL GRUPO 1: PASA
Filas esperadas: 60
Filas obtenidas: 60
Zonas: ['Z1', 'Z2', 'Z3', 'Z4', 'Z5']
Bloques: 12


,bloque,hora_inicio,zona,personas
0,0,0,Z1,400.0
1,0,0,Z2,2175.0
2,0,0,Z3,2400.0
3,0,0,Z4,1132.1
4,0,0,Z5,1500.0
5,1,6,Z1,400.0
6,1,6,Z2,2200.0
7,1,6,Z3,2400.0
8,1,6,Z4,1379.7
9,1,6,Z5,1500.0


In [25]:
datos_modelo_preintercambio = {
    clave: (
        valor.copy(deep=True)
        if isinstance(valor, pd.DataFrame)
        else valor
    )
    for clave, valor in datos_modelo.items()
}

demanda_formal_grupo1 = crear_tabla_demanda(
    poblacion=poblacion_grupo1[
        [
            "bloque",
            "zona",
            "personas",
        ]
    ],
    consumo=datos_modelo["consumo"],
    configuracion=configuracion_modelo,
    rng=np.random.default_rng(SEMILLA_BASE),
)

assert len(demanda_formal_grupo1) == (
    N_PASOS
    * len(ZONAS_ESPERADAS)
    * len(SUMINISTROS_ESPERADOS)
)

assert not demanda_formal_grupo1[
    [
        "bloque",
        "zona",
        "suministro",
        "demanda",
    ]
].isna().any().any()

assert (
    demanda_formal_grupo1["demanda"] >= 0
).all()

datos_modelo_post_grupo1 = {
    clave: (
        valor.copy(deep=True)
        if isinstance(valor, pd.DataFrame)
        else valor
    )
    for clave, valor in datos_modelo.items()
}

datos_modelo_post_grupo1[
    "demanda"
] = demanda_formal_grupo1

demanda_recalculada = (
    demanda_formal_grupo1["personas"]
    * demanda_formal_grupo1[
        "consumo_por_persona_bloque"
    ]
)

assert np.allclose(
    demanda_formal_grupo1["demanda"],
    demanda_recalculada,
)

resumen_demanda_grupo1 = (
    demanda_formal_grupo1
    .groupby(
        "suministro",
        as_index=False,
    )
    .agg(
        demanda_total=("demanda", "sum"),
        demanda_media_bloque_zona=(
            "demanda",
            "mean",
        ),
        demanda_maxima_bloque_zona=(
            "demanda",
            "max",
        ),
    )
)

print("DEMANDA FORMAL DEL GRUPO 1: PASA")
print(
    "Filas esperadas:",
    12 * 5 * 3,
)
print(
    "Filas obtenidas:",
    len(demanda_formal_grupo1),
)
print(
    "Datos previos al intercambio conservados:",
    datos_modelo_preintercambio[
        "demanda"
    ].empty,
)

display(demanda_formal_grupo1.head(15))
display(resumen_demanda_grupo1)

DEMANDA FORMAL DEL GRUPO 1: PASA
Filas esperadas: 180
Filas obtenidas: 180
Datos previos al intercambio conservados: True


,bloque,tiempo_h,zona,personas,suministro,consumo_por_persona_bloque,demanda,unidad
0,0,0,Z1,400.0,agua,0.7500,300.00000,litros/persona/bloque
1,0,0,Z1,400.0,alimentos,0.2500,100.00000,raciones/persona/bloque
2,0,0,Z1,400.0,medicamentos,0.0125,5.00000,kits/persona/bloque
3,0,0,Z2,2175.0,agua,0.7500,1631.25000,litros/persona/bloque
4,0,0,Z2,2175.0,alimentos,0.2500,543.75000,raciones/persona/bloque
5,0,0,Z2,2175.0,medicamentos,0.0125,27.18750,kits/persona/bloque
6,0,0,Z3,2400.0,agua,0.7500,1800.00000,litros/persona/bloque
7,0,0,Z3,2400.0,alimentos,0.2500,600.00000,raciones/persona/bloque
8,0,0,Z3,2400.0,medicamentos,0.0125,30.00000,kits/persona/bloque
9,0,0,Z4,1132.1,agua,0.7500,849.07500,litros/persona/bloque


,suministro,demanda_total,demanda_media_bloque_zona,demanda_maxima_bloque_zona
0,agua,75231.37500,1253.856250,1800.0
1,alimentos,25077.12500,417.952083,600.0
2,medicamentos,1253.85625,20.897604,30.0


### Accesibilidad recibida del Grupo 4

El Grupo 4 entregó la accesibilidad de cada zona durante los 12 bloques, incluyendo el tipo de vehículo recomendado, el tiempo de viaje desde Z4 y el índice medio de aislamiento.

La información está agregada por zona y no identifica las nueve rutas individuales del Grupo 3. Por ello, se utilizará para actualizar las condiciones generales de acceso a cada zona, pero no para inventar capacidades específicas de rutas.

El Grupo 4 utiliza `P` para camión de 10 toneladas y `L` para vehículo liviano de 2.5 toneladas. El vehículo liviano se relacionará provisionalmente con la camioneta de 2 toneladas del Grupo 3 por ser la categoría disponible más cercana. Esta equivalencia se conservará como una decisión explícita.

In [28]:
def extraer_tablas_docx(
    ruta_docx: str | Path,
) -> list[list[list[str]]]:

    ruta_docx = Path(ruta_docx)

    if not ruta_docx.exists():
        raise FileNotFoundError(
            f"No se encontró el documento: {ruta_docx.resolve()}"
        )

    espacio_nombres = {
        "w": (
            "http://schemas.openxmlformats.org/"
            "wordprocessingml/2006/main"
        )
    }

    with ZipFile(ruta_docx) as archivo_docx:
        try:
            xml_documento = archivo_docx.read(
                "word/document.xml"
            )
        except KeyError as error:
            raise ValueError(
                "El archivo no contiene un documento Word válido."
            ) from error

    raiz = ET.fromstring(xml_documento)

    tablas_extraidas = []

    for tabla_xml in raiz.findall(
        ".//w:tbl",
        espacio_nombres,
    ):
        filas_extraidas = []

        for fila_xml in tabla_xml.findall(
            "./w:tr",
            espacio_nombres,
        ):
            celdas_extraidas = []

            for celda_xml in fila_xml.findall(
                "./w:tc",
                espacio_nombres,
            ):
                fragmentos = [
                    nodo.text
                    for nodo in celda_xml.findall(
                        ".//w:t",
                        espacio_nombres,
                    )
                    if nodo.text is not None
                ]

                texto_celda = " ".join(
                    fragmentos
                ).strip()

                celdas_extraidas.append(
                    texto_celda
                )

            filas_extraidas.append(
                celdas_extraidas
            )

        tablas_extraidas.append(
            filas_extraidas
        )

    return tablas_extraidas


def cargar_accesibilidad_grupo4(
    ruta_docx: str | Path,
    configuracion: dict[str, Any],
) -> pd.DataFrame:

    tablas_documento = extraer_tablas_docx(
        ruta_docx
    )

    if len(tablas_documento) < 1:
        raise ValueError(
            "El reporte del Grupo 4 no contiene tablas."
        )

    tabla_accesibilidad = tablas_documento[0]

    if len(tabla_accesibilidad) != 13:
        raise ValueError(
            "Se esperaban 13 filas: encabezado y 12 bloques. "
            f"Se encontraron {len(tabla_accesibilidad)}."
        )

    encabezados = [
        texto.strip()
        for texto in tabla_accesibilidad[0]
    ]

    encabezados_esperados = [
        "Bloque",
        *configuracion["zonas_esperadas"],
    ]

    if encabezados != encabezados_esperados:
        raise ValueError(
            "Los encabezados del mapa de accesibilidad "
            "no coinciden con lo esperado. "
            f"Se encontró: {encabezados}"
        )

    mapeo_vehiculos = {
        "P": "camion_grande",
        "L": "camioneta",
    }

    patron_celda = re.compile(
        r"([PL])\s*·\s*"
        r"(\d+(?:\.\d+)?)\s*min\s*"
        r"I\s*=\s*(\d+(?:\.\d+)?)"
    )

    registros = []

    for fila in tabla_accesibilidad[1:]:
        hora_texto = fila[0].strip()

        coincidencia_hora = re.fullmatch(
            r"(\d+)\s*h",
            hora_texto,
        )

        if coincidencia_hora is None:
            raise ValueError(
                f"No se pudo interpretar el bloque: {hora_texto}"
            )

        hora_inicio = int(
            coincidencia_hora.group(1)
        )

        bloque = (
            hora_inicio
            // configuracion["paso_horas"]
        )

        for posicion, zona in enumerate(
            configuracion["zonas_esperadas"],
            start=1,
        ):
            texto_celda = (
                fila[posicion]
                .replace("\n", " ")
                .strip()
            )

            coincidencia = patron_celda.fullmatch(
                texto_celda
            )

            if coincidencia is None:
                raise ValueError(
                    "No se pudo interpretar la celda "
                    f"del bloque {bloque}, zona {zona}: "
                    f"'{texto_celda}'"
                )

            codigo_vehiculo = coincidencia.group(1)
            tiempo_viaje_min = float(
                coincidencia.group(2)
            )
            indice_aislamiento = float(
                coincidencia.group(3)
            )

            registros.append(
                {
                    "bloque": bloque,
                    "hora_inicio": hora_inicio,
                    "zona": zona,
                    "codigo_vehiculo_grupo4": (
                        codigo_vehiculo
                    ),
                    "vehiculo_recomendado": (
                        mapeo_vehiculos[
                            codigo_vehiculo
                        ]
                    ),
                    "tiempo_viaje_min": (
                        tiempo_viaje_min
                    ),
                    "tiempo_viaje_h": (
                        tiempo_viaje_min / 60
                    ),
                    "indice_aislamiento": (
                        indice_aislamiento
                    ),
                    "aislamiento_critico": (
                        indice_aislamiento < 0.30
                    ),
                    "fuente": (
                        "Grupo4_Reporte_Infraestructura.docx"
                    ),
                }
            )

    accesibilidad = pd.DataFrame(registros)

    filas_esperadas = (
        configuracion["n_pasos"]
        * len(configuracion["zonas_esperadas"])
    )

    if len(accesibilidad) != filas_esperadas:
        raise ValueError(
            f"Se esperaban {filas_esperadas} combinaciones "
            f"y se encontraron {len(accesibilidad)}."
        )

    if accesibilidad.duplicated(
        subset=["bloque", "zona"]
    ).any():
        raise ValueError(
            "Existen combinaciones duplicadas de bloque y zona."
        )

    if not accesibilidad[
        "indice_aislamiento"
    ].between(0, 1).all():
        raise ValueError(
            "El índice de aislamiento debe estar entre 0 y 1."
        )

    if (accesibilidad["tiempo_viaje_h"] < 0).any():
        raise ValueError(
            "Los tiempos de viaje no pueden ser negativos."
        )

    bloques_observados = set(
        accesibilidad["bloque"]
    )

    bloques_esperados = set(
        range(configuracion["n_pasos"])
    )

    if bloques_observados != bloques_esperados:
        raise ValueError(
            "Los bloques del Grupo 4 no coinciden con 0-11."
        )

    return (
        accesibilidad
        .sort_values(["bloque", "zona"])
        .reset_index(drop=True)
    )


accesibilidad_grupo4 = cargar_accesibilidad_grupo4(
    ruta_docx=RUTA_REPORTE_GRUPO4,
    configuracion=configuracion_modelo,
)

print("OUTPUT DEL GRUPO 4: PASA")
print("Filas esperadas:", 12 * 5)
print("Filas obtenidas:", len(accesibilidad_grupo4))
print(
    "Zonas:",
    accesibilidad_grupo4[
        "zona"
    ].unique().tolist(),
)
print(
    "Casos de aislamiento crítico:",
    int(
        accesibilidad_grupo4[
            "aislamiento_critico"
        ].sum()
    ),
)

display(accesibilidad_grupo4.head(10))

OUTPUT DEL GRUPO 4: PASA
Filas esperadas: 60
Filas obtenidas: 60
Zonas: ['Z1', 'Z2', 'Z3', 'Z4', 'Z5']
Casos de aislamiento crítico: 2


,bloque,hora_inicio,zona,codigo_vehiculo_grupo4,vehiculo_recomendado,tiempo_viaje_min,tiempo_viaje_h,indice_aislamiento,aislamiento_critico,fuente
0,0,0,Z1,P,camion_grande,16.0,0.266667,0.59,False,Grupo4_Reporte_Infraestructura.docx
1,0,0,Z2,L,camioneta,29.0,0.483333,0.58,False,Grupo4_Reporte_Infraestructura.docx
2,0,0,Z3,L,camioneta,50.0,0.833333,0.44,False,Grupo4_Reporte_Infraestructura.docx
3,0,0,Z4,P,camion_grande,0.0,0.000000,0.94,False,Grupo4_Reporte_Infraestructura.docx
4,0,0,Z5,L,camioneta,81.0,1.350000,0.26,True,Grupo4_Reporte_Infraestructura.docx
5,1,6,Z1,P,camion_grande,16.0,0.266667,0.59,False,Grupo4_Reporte_Infraestructura.docx
6,1,6,Z2,L,camioneta,29.0,0.483333,0.58,False,Grupo4_Reporte_Infraestructura.docx
7,1,6,Z3,L,camioneta,50.0,0.833333,0.44,False,Grupo4_Reporte_Infraestructura.docx
8,1,6,Z4,P,camion_grande,0.0,0.000000,0.94,False,Grupo4_Reporte_Infraestructura.docx
9,1,6,Z5,L,camioneta,81.0,1.350000,0.26,True,Grupo4_Reporte_Infraestructura.docx


In [29]:
def integrar_accesibilidad_en_rutas(
    rutas_simulacion: pd.DataFrame,
    accesibilidad: pd.DataFrame,
) -> pd.DataFrame:

    columnas_rutas = {
        "bloque",
        "ruta_id",
        "centro",
        "destino",
        "zona",
        "tiempo_transito_h",
        "capacidad_vehiculos_viaje",
        "ruta_disponible",
        "fuente_estado_ruta",
    }

    columnas_accesibilidad = {
        "bloque",
        "zona",
        "vehiculo_recomendado",
        "tiempo_viaje_h",
        "indice_aislamiento",
        "aislamiento_critico",
    }

    faltantes_rutas = columnas_rutas.difference(
        rutas_simulacion.columns
    )

    faltantes_accesibilidad = (
        columnas_accesibilidad.difference(
            accesibilidad.columns
        )
    )

    if faltantes_rutas:
        raise ValueError(
            "Faltan columnas en las rutas: "
            f"{sorted(faltantes_rutas)}"
        )

    if faltantes_accesibilidad:
        raise ValueError(
            "Faltan columnas de accesibilidad: "
            f"{sorted(faltantes_accesibilidad)}"
        )

    actualizacion_zonal = accesibilidad[
        [
            "bloque",
            "zona",
            "vehiculo_recomendado",
            "tiempo_viaje_h",
            "indice_aislamiento",
            "aislamiento_critico",
        ]
    ].rename(
        columns={
            "tiempo_viaje_h": (
                "tiempo_grupo4_desde_z4_h"
            ),
        }
    )

    rutas_integradas = rutas_simulacion.merge(
        actualizacion_zonal,
        on=["bloque", "zona"],
        how="left",
        validate="many_to_one",
    )

    rutas_integradas[
        "tiempo_transito_antes_intercambio_h"
    ] = rutas_integradas[
        "tiempo_transito_h"
    ]

    rutas_integradas[
        "capacidad_antes_intercambio"
    ] = rutas_integradas[
        "capacidad_vehiculos_viaje"
    ]

    actualizar_tiempo = (
        rutas_integradas["centro"].eq(
            "Bodega Central CONRED"
        )
        & rutas_integradas[
            "tiempo_grupo4_desde_z4_h"
        ].notna()
    )

    rutas_integradas.loc[
        actualizar_tiempo,
        "tiempo_transito_h",
    ] = rutas_integradas.loc[
        actualizar_tiempo,
        "tiempo_grupo4_desde_z4_h",
    ]

    rutas_integradas.loc[
        actualizar_tiempo,
        "fuente_estado_ruta",
    ] = (
        "Tiempo actualizado con output formal del Grupo 4"
    )

    rutas_integradas[
        "tiempo_actualizado_grupo4"
    ] = actualizar_tiempo

    rutas_integradas[
        "acceso_zonal_reportado"
    ] = (
        rutas_integradas[
            "vehiculo_recomendado"
        ].notna()
        & rutas_integradas[
            "tiempo_grupo4_desde_z4_h"
        ].notna()
    )

    capacidad_sin_inventar = (
        rutas_integradas[
            "capacidad_vehiculos_viaje"
        ]
        ==
        rutas_integradas[
            "capacidad_antes_intercambio"
        ]
    ).all()

    if not capacidad_sin_inventar:
        raise RuntimeError(
            "Se modificó una capacidad sin información formal."
        )

    return (
        rutas_integradas
        .sort_values(["bloque", "ruta_id"])
        .reset_index(drop=True)
    )


rutas_post_intercambio = integrar_accesibilidad_en_rutas(
    rutas_simulacion=datos_modelo[
        "rutas_simulacion"
    ],
    accesibilidad=accesibilidad_grupo4,
)

datos_modelo_post_intercambio = {
    clave: (
        valor.copy(deep=True)
        if isinstance(valor, pd.DataFrame)
        else valor
    )
    for clave, valor in datos_modelo_post_grupo1.items()
}

datos_modelo_post_intercambio[
    "rutas_simulacion"
] = rutas_post_intercambio

configuracion_post_intercambio = {
    **configuracion_modelo,
    "nombre_escenario": "post_intercambio",
}

print("INTEGRACIÓN DE ACCESIBILIDAD: PASA")
print(
    "Filas de rutas:",
    len(rutas_post_intercambio),
)
print(
    "Tiempos actualizados desde Z4:",
    int(
        rutas_post_intercambio[
            "tiempo_actualizado_grupo4"
        ].sum()
    ),
)
print(
    "Rutas-bloque sin zona identificada:",
    int(
        rutas_post_intercambio[
            "zona"
        ].isna().sum()
    ),
)
print(
    "Capacidades modificadas:",
    int(
        (
            rutas_post_intercambio[
                "capacidad_vehiculos_viaje"
            ]
            !=
            rutas_post_intercambio[
                "capacidad_antes_intercambio"
            ]
        ).sum()
    ),
)

display(
    rutas_post_intercambio[
        [
            "bloque",
            "ruta_id",
            "centro",
            "destino",
            "zona",
            "ruta_disponible",
            "tiempo_transito_antes_intercambio_h",
            "tiempo_transito_h",
            "vehiculo_recomendado",
            "indice_aislamiento",
            "aislamiento_critico",
            "tiempo_actualizado_grupo4",
        ]
    ].head(18)
)

INTEGRACIÓN DE ACCESIBILIDAD: PASA
Filas de rutas: 108
Tiempos actualizados desde Z4: 36
Rutas-bloque sin zona identificada: 24
Capacidades modificadas: 0


,bloque,ruta_id,centro,destino,zona,ruta_disponible,tiempo_transito_antes_intercambio_h,tiempo_transito_h,vehiculo_recomendado,indice_aislamiento,aislamiento_critico,tiempo_actualizado_grupo4
0,0,R01,Bodega Central CONRED,Albergue Estadio,NaN,True,0.40,0.400000,NaN,NaN,NaN,False
1,0,R02,Bodega Central CONRED,Albergue Z2,Z2,True,0.90,0.483333,camioneta,0.58,False,True
2,0,R03,Bodega Central CONRED,Albergue Z5-1,Z5,True,7.00,1.350000,camioneta,0.26,True,True
3,0,R04,Bodega Central CONRED,Albergue Z5-2,Z5,True,8.75,1.350000,camioneta,0.26,True,True
4,0,R05,Almacén Norte,Albergue Z2,Z2,True,0.20,0.200000,camioneta,0.58,False,False
5,0,R06,Almacén Norte,Albergue Estadio,NaN,True,0.70,0.700000,NaN,NaN,NaN,False
6,0,R07,Depósito Sur,Albergue Z3,Z3,True,0.40,0.400000,camioneta,0.44,False,False
7,0,R08,Centro Logístico Z5,Albergue Z5-1,Z5,False,NaN,NaN,camioneta,0.26,True,False
8,0,R09,Centro Logístico Z5,Albergue Z5-2,Z5,False,NaN,NaN,camioneta,0.26,True,False
9,1,R01,Bodega Central CONRED,Albergue Estadio,NaN,True,0.40,0.400000,NaN,NaN,NaN,False


In [30]:
configuracion_preintercambio = {
    **configuracion_modelo,
    "nombre_escenario": "preintercambio_tecnico",
    "incertidumbre": {
        **configuracion_modelo["incertidumbre"],
    },
}

configuracion_post_intercambio = {
    **configuracion_modelo,
    "nombre_escenario": "post_intercambio",
    "incertidumbre": {
        "cv_demanda": 0.10,
        "cv_tiempo_ruta": 0.10,
        "cv_reposicion": 0.05,
    },
}

datos_modelo_post_intercambio[
    "poblacion"
] = poblacion_grupo1[
    [
        "bloque",
        "zona",
        "personas",
    ]
].copy(deep=True)

for nombre, valor in configuracion_post_intercambio[
    "incertidumbre"
].items():
    if not 0 <= valor < 1:
        raise ValueError(
            f"{nombre} debe encontrarse entre 0 y 1."
        )

print("CONFIGURACIÓN DE INCERTIDUMBRE: PASA")
print("Escenario previo:")
print(
    configuracion_preintercambio[
        "incertidumbre"
    ]
)
print("Escenario posterior:")
print(
    configuracion_post_intercambio[
        "incertidumbre"
    ]
)

CONFIGURACIÓN DE INCERTIDUMBRE: PASA
Escenario previo:
{'cv_demanda': 0.0, 'cv_tiempo_ruta': 0.0, 'cv_reposicion': 0.0}
Escenario posterior:
{'cv_demanda': 0.1, 'cv_tiempo_ruta': 0.1, 'cv_reposicion': 0.05}


## Carga del Excel

In [2]:
def cargar_datos_grupo3(
    ruta_excel: str | Path
) -> dict[str, pd.DataFrame]:

    ruta_excel = Path(ruta_excel)

    if not ruta_excel.exists():
        raise FileNotFoundError(
            f"No se encontró el Excel del Grupo 3 en: "
            f"{ruta_excel.resolve()}"
        )

    archivo_excel = pd.ExcelFile(ruta_excel)
    nombres_hojas = archivo_excel.sheet_names

    if HOJA_DATOS not in nombres_hojas:
        raise ValueError(
            f"Falta la hoja requerida '{HOJA_DATOS}'. "
            f"Hojas encontradas: {nombres_hojas}"
        )

    hoja_completa = pd.read_excel(
        archivo_excel,
        sheet_name=HOJA_DATOS,
        header=None,
    )

    encabezados_secciones = {
        "centros": "1. CENTROS DE ACOPIO",
        "consumo": "2. TASAS DE CONSUMO",
        "rutas": "3. RED DE DISTRIBUCIÓN",
        "flota": "4. FLOTA DE VEHÍCULOS",
        "reposicion": "5. REPOSICIÓN DE SUMINISTROS",
        "output_requerido": "OUTPUT REQUERIDO",
    }

    texto_primera_columna = (
        hoja_completa.iloc[:, 0]
        .fillna("")
        .astype(str)
    )

    filas_secciones = {}

    for nombre, texto_buscado in encabezados_secciones.items():
        coincidencias = texto_primera_columna.str.contains(
            texto_buscado,
            case=False,
            regex=False,
        )

        filas_encontradas = hoja_completa.index[
            coincidencias
        ].tolist()

        filas_secciones[nombre] = (
            filas_encontradas[0]
            if filas_encontradas
            else None
        )

    print(f"Archivo cargado: {ruta_excel.name}")
    print(f"Hojas encontradas: {nombres_hojas}")
    print(
        f"Dimensiones de '{HOJA_DATOS}': "
        f"{hoja_completa.shape}"
    )
    print(
        "Filas iniciales de las secciones:",
        filas_secciones,
    )

    return {
        "hoja_completa": hoja_completa.copy(deep=True),
        "filas_secciones": pd.DataFrame(
            list(filas_secciones.items()),
            columns=["seccion", "fila_inicio"],
        ),
    }


datos_crudos = cargar_datos_grupo3(RUTA_EXCEL)

Archivo cargado: Grupo3_CadenaSuministro.xlsx
Hojas encontradas: ['Datos_Grupo3']
Dimensiones de 'Datos_Grupo3': (53, 6)
Filas iniciales de las secciones: {'centros': 4, 'consumo': 11, 'rutas': 17, 'flota': 29, 'reposicion': 35, 'output_requerido': 48}


## Normalización de tablas

### Centro de acopio

In [3]:
def normalizar_centros(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de centros de acopio."
        )

    fila_encabezados = fila_inicio + 1
    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 4

    centros = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:6,
    ].copy()

    centros.columns = [
        "centro",
        "zona_original",
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    centros["zona"] = centros[
        "zona_original"
    ].str.extract(r"(Z\d+)", expand=False)

    columnas_numericas = [
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    for columna in columnas_numericas:
        centros[columna] = pd.to_numeric(
            centros[columna],
            errors="raise",
        )

    orden_columnas = [
        "centro",
        "zona",
        "zona_original",
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    return centros[orden_columnas].reset_index(drop=True)


fila_centros = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["centros", "fila_inicio"]
)

df_centros = normalizar_centros(
    datos_crudos["hoja_completa"],
    fila_centros,
)

display(df_centros)

,centro,zona,zona_original,agua_litros,alimentos_raciones,medicamentos_kits,capacidad_almacenamiento_m3
0,Bodega Central CONRED,Z4,Z4 (Este comercial),180000,42000,1200,2400
1,Almacén Norte,Z2,Z2 (Norte residencial),65000,18000,380,900
2,Depósito Sur,Z3,Z3 (Sur industrial),40000,9500,210,550
3,Centro Logístico Z5,Z5,Z5 (Oeste periférico),28000,7200,145,400


### Tasas de consumo

In [4]:
def normalizar_consumo(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
    paso_horas: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de tasas de consumo."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 3

    consumo = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:4,
    ].copy()

    consumo.columns = [
        "suministro_original",
        "consumo_por_persona_dia",
        "unidad_original",
        "nota",
    ]

    nombres_normalizados = {
        "Agua potable": "agua",
        "Alimentos (raciones)": "alimentos",
        "Medicamentos básicos": "medicamentos",
    }

    consumo["suministro"] = consumo[
        "suministro_original"
    ].map(nombres_normalizados)

    consumo["consumo_por_persona_dia"] = pd.to_numeric(
        consumo["consumo_por_persona_dia"],
        errors="raise",
    )

    consumo["consumo_por_persona_bloque"] = (
        consumo["consumo_por_persona_dia"]
        * paso_horas
        / 24
    )

    unidades_por_bloque = {
        "agua": "litros/persona/bloque",
        "alimentos": "raciones/persona/bloque",
        "medicamentos": "kits/persona/bloque",
    }

    consumo["unidad_por_bloque"] = consumo[
        "suministro"
    ].map(unidades_por_bloque)

    orden_columnas = [
        "suministro",
        "suministro_original",
        "consumo_por_persona_dia",
        "unidad_original",
        "consumo_por_persona_bloque",
        "unidad_por_bloque",
        "nota",
    ]

    return consumo[orden_columnas].reset_index(drop=True)


fila_consumo = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["consumo", "fila_inicio"]
)

df_consumo = normalizar_consumo(
    datos_crudos["hoja_completa"],
    fila_consumo,
    PASO_HORAS,
)

display(df_consumo)

,suministro,suministro_original,consumo_por_persona_dia,unidad_original,consumo_por_persona_bloque,unidad_por_bloque,nota
0,agua,Agua potable,3.00,litros,0.7500,litros/persona/bloque,Mínimo OPS en emergencia
1,alimentos,Alimentos (raciones),1.00,ración/día,0.2500,raciones/persona/bloque,Ración = 2100 kcal
2,medicamentos,Medicamentos básicos,0.05,kit/persona,0.0125,kits/persona/bloque,1 kit cubre 20 personas/día


### Red de distribución

In [5]:
def normalizar_rutas(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de la red de distribución."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 9

    rutas = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    rutas.columns = [
        "ruta_original",
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
        "estado_ruta",
    ]

    componentes_ruta = rutas["ruta_original"].str.split(
        r"\s*(?:→|->)\s*",
        n=1,
        expand=True,
        regex=True,
    )

    rutas["origen"] = componentes_ruta[0].str.strip()
    rutas["destino"] = componentes_ruta[1].str.strip()

    columnas_numericas = [
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
    ]

    for columna in columnas_numericas:
        rutas[columna] = pd.to_numeric(
            rutas[columna],
            errors="raise",
        )

    rutas["ruta_id"] = [
        f"R{i:02d}"
        for i in range(1, len(rutas) + 1)
    ]

    orden_columnas = [
        "ruta_id",
        "ruta_original",
        "origen",
        "destino",
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
        "estado_ruta",
    ]

    return rutas[orden_columnas].reset_index(drop=True)


fila_rutas = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["rutas", "fila_inicio"]
)

df_rutas = normalizar_rutas(
    datos_crudos["hoja_completa"],
    fila_rutas,
)

display(df_rutas)

,ruta_id,ruta_original,origen,destino,distancia_km,tiempo_normal_h,capacidad_vehiculos_viaje,estado_ruta
0,R01,Bodega Central → Albergue Estadio,Bodega Central,Albergue Estadio,4.2,0.4,8,Disponible
1,R02,Bodega Central → Albergue Z2,Bodega Central,Albergue Z2,7.8,0.9,6,Disponible
2,R03,Bodega Central → Albergue Z5-1,Bodega Central,Albergue Z5-1,12.1,2.8,4,Dañada — velocidad reducida 60%
3,R04,Bodega Central → Albergue Z5-2,Bodega Central,Albergue Z5-2,13.4,3.5,3,Dañada — velocidad reducida 60%
4,R05,Almacén Norte → Albergue Z2,Almacén Norte,Albergue Z2,2.1,0.2,10,Disponible
5,R06,Almacén Norte → Albergue Estadio,Almacén Norte,Albergue Estadio,6.4,0.7,7,Disponible
6,R07,Depósito Sur → Albergue Z3,Depósito Sur,Albergue Z3,3.8,0.4,8,Disponible
7,R08,Centro Log. Z5 → Albergue Z5-1,Centro Log. Z5,Albergue Z5-1,1.2,0.5,6,Parcialmente bloqueada
8,R09,Centro Log. Z5 → Albergue Z5-2,Centro Log. Z5,Albergue Z5-2,2.4,0.9,5,Parcialmente bloqueada


### Flota de vehículos

In [6]:
def normalizar_flota(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de la flota de vehículos."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 3

    flota = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    flota.columns = [
        "tipo_vehiculo_original",
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    nombres_normalizados = {
        "Camión grande (10 ton)": "camion_grande",
        "Camioneta (2 ton)": "camioneta",
        "Motocicleta (mensajería)": "motocicleta",
    }

    flota["tipo_vehiculo"] = flota[
        "tipo_vehiculo_original"
    ].map(nombres_normalizados)

    columnas_numericas = [
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    for columna in columnas_numericas:
        flota[columna] = pd.to_numeric(
            flota[columna],
            errors="raise",
        )

    flota["cantidad_disponible"] = flota[
        "cantidad_disponible"
    ].astype(int)

    orden_columnas = [
        "tipo_vehiculo",
        "tipo_vehiculo_original",
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    return flota[orden_columnas].reset_index(drop=True)


fila_flota = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["flota", "fila_inicio"]
)

df_flota = normalizar_flota(
    datos_crudos["hoja_completa"],
    fila_flota,
)

display(df_flota)

,tipo_vehiculo,tipo_vehiculo_original,cantidad_disponible,capacidad_m3,consumo_combustible_gal_km,combustible_disponible_gal
0,camion_grande,Camión grande (10 ton),8,28.0,0.18,1400
1,camioneta,Camioneta (2 ton),22,8.0,0.09,1800
2,motocicleta,Motocicleta (mensajería),15,0.3,0.03,420


### Reposiciones

In [7]:
def normalizar_reposicion(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de reposición de suministros."
        )

    primera_fila_datos = fila_inicio + 2

    ultima_fila_datos = primera_fila_datos + 6

    reposicion = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    reposicion.columns = [
        "bloque_original",
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
        "fuente",
    ]

    componentes_tiempo = reposicion[
        "bloque_original"
    ].str.extract(
        r"T(\d+)\s*\((\d+)-(\d+)h\)"
    )

    componentes_tiempo.columns = [
        "bloque",
        "hora_inicio",
        "hora_fin",
    ]

    reposicion[
        ["bloque", "hora_inicio", "hora_fin"]
    ] = componentes_tiempo.astype(int)

    columnas_numericas = [
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
    ]

    for columna in columnas_numericas:
        reposicion[columna] = pd.to_numeric(
            reposicion[columna],
            errors="raise",
        )

    orden_columnas = [
        "bloque",
        "bloque_original",
        "hora_inicio",
        "hora_fin",
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
        "fuente",
    ]

    return reposicion[orden_columnas].reset_index(drop=True)


fila_reposicion = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["reposicion", "fila_inicio"]
)

df_reposicion = normalizar_reposicion(
    datos_crudos["hoja_completa"],
    fila_reposicion,
)

display(df_reposicion)

,bloque,bloque_original,hora_inicio,hora_fin,agua_adicional_litros,alimentos_adicionales_raciones,medicamentos_adicionales_kits,fuente
0,2,T2 (12-18h),12,18,0,0,0,Sin reposición aún
1,3,T3 (18-24h),18,24,40000,8000,200,Cruz Roja — primer convoy
2,4,T4 (24-30h),24,30,80000,15000,400,Gobierno central
3,6,T6 (36-42h),36,42,120000,22000,600,Ayuda internacional
4,8,T8 (48-54h),48,54,150000,28000,800,Ayuda internacional
5,10,T10 (60-66h),60,66,100000,18000,500,Cruz Roja — segundo convoy


### Output requerido para Grupo 7

In [9]:
def normalizar_output_requerido(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de output requerido."
        )

    primera_fila_datos = fila_inicio + 2

    output_requerido = (
        hoja_completa.iloc[primera_fila_datos:, 0]
        .dropna()
        .astype(str)
        .str.strip()
    )

    output_requerido = output_requerido[
        output_requerido.ne("")
    ].reset_index(drop=True)

    if len(output_requerido) != 3:
        raise ValueError(
            "Se esperaban 3 outputs requeridos, pero se "
            f"encontraron {len(output_requerido)}."
        )

    nombres_output = [
        "balance_suministros",
        "primer_agotamiento",
        "recomendacion_logistica",
    ]

    tabla_output = pd.DataFrame(
        {
            "output_id": [
                "O01",
                "O02",
                "O03",
            ],
            "nombre_output": nombres_output,
            "descripcion_original": output_requerido,
        }
    )

    return tabla_output


fila_output_requerido = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["output_requerido", "fila_inicio"]
)

df_output_requerido = normalizar_output_requerido(
    datos_crudos["hoja_completa"],
    fila_output_requerido,
)

display(df_output_requerido)

,output_id,nombre_output,descripcion_original
0,O01,balance_suministros,a) Balance de suministros por bloque de tiempo...
1,O02,primer_agotamiento,b) Identificación del momento en que algún sum...
2,O03,recomendacion_logistica,c) Recomendación de priorización de rutas y fr...


### Función de normalización

In [10]:
def normalizar_datos_grupo3(
    datos: dict[str, pd.DataFrame],
) -> dict[str, pd.DataFrame]:

    claves_requeridas = {
        "hoja_completa",
        "filas_secciones",
    }

    claves_faltantes = claves_requeridas.difference(
        datos.keys()
    )

    if claves_faltantes:
        raise KeyError(
            "Faltan estructuras necesarias para normalizar: "
            f"{sorted(claves_faltantes)}"
        )

    hoja_completa = datos["hoja_completa"]

    filas_secciones = (
        datos["filas_secciones"]
        .set_index("seccion")["fila_inicio"]
        .to_dict()
    )

    secciones_requeridas = [
        "centros",
        "consumo",
        "rutas",
        "flota",
        "reposicion",
        "output_requerido",
    ]

    secciones_no_encontradas = [
        seccion
        for seccion in secciones_requeridas
        if pd.isna(filas_secciones.get(seccion))
    ]

    if secciones_no_encontradas:
        raise ValueError(
            "No se localizaron las siguientes secciones: "
            f"{secciones_no_encontradas}"
        )

    df_centros_normalizado = normalizar_centros(
        hoja_completa,
        int(filas_secciones["centros"]),
    )

    df_consumo_normalizado = normalizar_consumo(
        hoja_completa,
        int(filas_secciones["consumo"]),
        PASO_HORAS,
    )

    df_rutas_normalizado = normalizar_rutas(
        hoja_completa,
        int(filas_secciones["rutas"]),
    )

    df_flota_normalizado = normalizar_flota(
        hoja_completa,
        int(filas_secciones["flota"]),
    )

    df_reposicion_normalizado = normalizar_reposicion(
        hoja_completa,
        int(filas_secciones["reposicion"]),
    )

    df_output_normalizado = normalizar_output_requerido(
        hoja_completa,
        int(filas_secciones["output_requerido"]),
    )

    return {
        "centros": df_centros_normalizado,
        "consumo": df_consumo_normalizado,
        "rutas": df_rutas_normalizado,
        "flota": df_flota_normalizado,
        "reposicion": df_reposicion_normalizado,
        "output_requerido": df_output_normalizado,
    }


datos_modelo = normalizar_datos_grupo3(datos_crudos)

resumen_tablas = pd.DataFrame(
    {
        "tabla": datos_modelo.keys(),
        "filas": [
            len(tabla)
            for tabla in datos_modelo.values()
        ],
        "columnas": [
            len(tabla.columns)
            for tabla in datos_modelo.values()
        ],
    }
)

display(resumen_tablas)

,tabla,filas,columnas
0,centros,4,7
1,consumo,3,7
2,rutas,9,8
3,flota,3,6
4,reposicion,6,8
5,output_requerido,3,3


## Validación de datos

In [11]:
def validar_datos_grupo3(
    datos: dict[str, pd.DataFrame],
) -> None:

    errores = []
    advertencias = []
    pruebas_superadas = []

    tablas_requeridas = {
        "centros",
        "consumo",
        "rutas",
        "flota",
        "reposicion",
        "output_requerido",
    }

    tablas_faltantes = tablas_requeridas.difference(datos.keys())

    if tablas_faltantes:
        errores.append(
            f"Faltan tablas requeridas: {sorted(tablas_faltantes)}"
        )

    if errores:
        raise ValueError("\n".join(errores))

    centros = datos["centros"]
    consumo = datos["consumo"]
    rutas = datos["rutas"]
    flota = datos["flota"]
    reposicion = datos["reposicion"]
    output_requerido = datos["output_requerido"]

    cantidades_esperadas = {
        "centros": (len(centros), 4),
        "suministros": (len(consumo), 3),
        "rutas": (len(rutas), 9),
        "tipos de vehículo": (len(flota), 3),
        "reposiciones": (len(reposicion), 6),
        "outputs requeridos": (len(output_requerido), 3),
    }

    for nombre, (cantidad_real, cantidad_esperada) in (
        cantidades_esperadas.items()
    ):
        if cantidad_real != cantidad_esperada:
            errores.append(
                f"{nombre}: se esperaban {cantidad_esperada} "
                f"registros y se encontraron {cantidad_real}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre}: {cantidad_real} registros"
            )

    columnas_obligatorias = {
        "centros": [
            "centro",
            "zona",
            "agua_litros",
            "alimentos_raciones",
            "medicamentos_kits",
            "capacidad_almacenamiento_m3",
        ],
        "consumo": [
            "suministro",
            "consumo_por_persona_dia",
            "consumo_por_persona_bloque",
        ],
        "rutas": [
            "ruta_id",
            "origen",
            "destino",
            "distancia_km",
            "tiempo_normal_h",
            "capacidad_vehiculos_viaje",
            "estado_ruta",
        ],
        "flota": [
            "tipo_vehiculo",
            "cantidad_disponible",
            "capacidad_m3",
            "consumo_combustible_gal_km",
            "combustible_disponible_gal",
        ],
        "reposicion": [
            "bloque",
            "hora_inicio",
            "hora_fin",
            "agua_adicional_litros",
            "alimentos_adicionales_raciones",
            "medicamentos_adicionales_kits",
            "fuente",
        ],
        "output_requerido": [
            "output_id",
            "nombre_output",
            "descripcion_original",
        ],
    }

    for nombre_tabla, columnas in columnas_obligatorias.items():
        tabla = datos[nombre_tabla]

        faltantes = tabla[columnas].isna().sum()
        faltantes = faltantes[faltantes > 0]

        if not faltantes.empty:
            errores.append(
                f"{nombre_tabla}: valores faltantes en "
                f"{faltantes.to_dict()}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre_tabla}: sin valores faltantes obligatorios"
            )

    identificadores = {
        "centros": "centro",
        "consumo": "suministro",
        "rutas": "ruta_id",
        "flota": "tipo_vehiculo",
        "reposicion": "bloque",
        "output_requerido": "output_id",
    }

    for nombre_tabla, identificador in identificadores.items():
        tabla = datos[nombre_tabla]

        if tabla[identificador].duplicated().any():
            valores_duplicados = tabla.loc[
                tabla[identificador].duplicated(keep=False),
                identificador,
            ].tolist()

            errores.append(
                f"{nombre_tabla}: identificadores duplicados "
                f"{valores_duplicados}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre_tabla}: sin identificadores duplicados"
            )

    columnas_no_negativas = {
        "centros": [
            "agua_litros",
            "alimentos_raciones",
            "medicamentos_kits",
            "capacidad_almacenamiento_m3",
        ],
        "consumo": [
            "consumo_por_persona_dia",
            "consumo_por_persona_bloque",
        ],
        "flota": [
            "cantidad_disponible",
            "capacidad_m3",
            "consumo_combustible_gal_km",
            "combustible_disponible_gal",
        ],
        "reposicion": [
            "agua_adicional_litros",
            "alimentos_adicionales_raciones",
            "medicamentos_adicionales_kits",
        ],
    }

    for nombre_tabla, columnas in columnas_no_negativas.items():
        tabla = datos[nombre_tabla]

        for columna in columnas:
            if (tabla[columna] < 0).any():
                errores.append(
                    f"{nombre_tabla}.{columna} contiene "
                    "valores negativos."
                )

    columnas_positivas_rutas = [
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
    ]

    for columna in columnas_positivas_rutas:
        if (rutas[columna] <= 0).any():
            errores.append(
                f"rutas.{columna} contiene valores menores "
                "o iguales a cero."
            )

    bloques_esperados_reposicion = {2, 3, 4, 6, 8, 10}
    bloques_observados = set(reposicion["bloque"])

    if bloques_observados != bloques_esperados_reposicion:
        errores.append(
            "Los bloques de reposición no coinciden con "
            f"{sorted(bloques_esperados_reposicion)}."
        )
    else:
        pruebas_superadas.append(
            "bloques de reposición: T2, T3, T4, T6, T8 y T10"
        )

    duracion_reposiciones = (
        reposicion["hora_fin"] - reposicion["hora_inicio"]
    )

    if not (duracion_reposiciones == PASO_HORAS).all():
        errores.append(
            "Algún intervalo de reposición no dura "
            f"{PASO_HORAS} horas."
        )
    else:
        pruebas_superadas.append(
            "intervalos de reposición: 6 horas"
        )

    if not (
        reposicion["hora_inicio"]
        == reposicion["bloque"] * PASO_HORAS
    ).all():
        errores.append(
            "La hora inicial no coincide con bloque × PASO_HORAS."
        )
    else:
        pruebas_superadas.append(
            "correspondencia correcta entre bloques y horas"
        )

    consumo_esperado_bloque = (
        consumo["consumo_por_persona_dia"]
        * PASO_HORAS
        / 24
    )

    if not np.allclose(
        consumo["consumo_por_persona_bloque"],
        consumo_esperado_bloque,
    ):
        errores.append(
            "Las tasas de consumo por bloque no coinciden con "
            "las tasas diarias convertidas a seis horas."
        )
    else:
        pruebas_superadas.append(
            "tasas por bloque derivadas correctamente"
        )

    estados_permitidos = {
        "Disponible",
        "Dañada — velocidad reducida 60%",
        "Parcialmente bloqueada",
    }

    estados_desconocidos = (
        set(rutas["estado_ruta"]) - estados_permitidos
    )

    if estados_desconocidos:
        errores.append(
            f"Estados de ruta no reconocidos: {estados_desconocidos}."
        )
    else:
        pruebas_superadas.append(
            "estados de ruta originales preservados"
        )

    nombres_centros = set(centros["centro"])
    origenes_rutas = set(rutas["origen"])
    origenes_sin_coincidencia = origenes_rutas - nombres_centros

    if origenes_sin_coincidencia:
        advertencias.append(
            "Algunos nombres de origen no coinciden literalmente "
            "con la tabla de centros: "
            f"{sorted(origenes_sin_coincidencia)}. "
            "Se necesitará un mapeo explícito."
        )

    if errores:
        mensaje = "\n".join(
            f"- {error}" for error in errores
        )
        raise ValueError(
            "La validación encontró errores críticos:\n"
            f"{mensaje}"
        )

    print("VALIDACIÓN GENERAL: PASA")

    print("\nPruebas superadas:")
    for prueba in pruebas_superadas:
        print(f"  PASA — {prueba}")

    if advertencias:
        print("\nAdvertencias que requieren documentación:")
        for advertencia in advertencias:
            print(f"  ADVERTENCIA — {advertencia}")


validar_datos_grupo3(datos_modelo)

VALIDACIÓN GENERAL: PASA

Pruebas superadas:
  PASA — centros: 4 registros
  PASA — suministros: 3 registros
  PASA — rutas: 9 registros
  PASA — tipos de vehículo: 3 registros
  PASA — reposiciones: 6 registros
  PASA — outputs requeridos: 3 registros
  PASA — centros: sin valores faltantes obligatorios
  PASA — consumo: sin valores faltantes obligatorios
  PASA — rutas: sin valores faltantes obligatorios
  PASA — flota: sin valores faltantes obligatorios
  PASA — reposicion: sin valores faltantes obligatorios
  PASA — output_requerido: sin valores faltantes obligatorios
  PASA — centros: sin identificadores duplicados
  PASA — consumo: sin identificadores duplicados
  PASA — rutas: sin identificadores duplicados
  PASA — flota: sin identificadores duplicados
  PASA — reposicion: sin identificadores duplicados
  PASA — output_requerido: sin identificadores duplicados
  PASA — bloques de reposición: T2, T3, T4, T6, T8 y T10
  PASA — intervalos de reposición: 6 horas
  PASA — correspond

### Tabla de trazabilidad de los datos

In [12]:
tabla_trazabilidad = pd.DataFrame(
    [
        {
            "dato": "Inventario inicial de agua",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "litros",
            "uso_en_modelo": (
                "Estado inicial del inventario de agua por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Inventario inicial de alimentos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "raciones",
            "uso_en_modelo": (
                "Estado inicial del inventario de alimentos por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Inventario inicial de medicamentos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "kits",
            "uso_en_modelo": (
                "Estado inicial del inventario de medicamentos por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Capacidad de almacenamiento",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "m³",
            "uso_en_modelo": (
                "Referencia de capacidad física de cada centro"
            ),
            "transformacion": (
                "No se aplica todavía como restricción porque faltan "
                "factores de conversión de suministros a m³"
            ),
        },
        {
            "dato": "Consumo per cápita",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Tasas de consumo",
            "unidad": "unidad de suministro/persona/día",
            "uso_en_modelo": (
                "Cálculo de demanda de agua, alimentos y medicamentos"
            ),
            "transformacion": (
                "consumo_diario × PASO_HORAS / 24"
            ),
        },
        {
            "dato": "Distancia de las rutas",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "km",
            "uso_en_modelo": (
                "Cálculo de tiempo logístico y consumo de combustible"
            ),
            "transformacion": (
                "Valor sin modificar; falta definir si se considera "
                "viaje de ida o de ida y vuelta"
            ),
        },
        {
            "dato": "Tiempo normal de ruta",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "horas",
            "uso_en_modelo": (
                "Programación de llegadas de los vehículos"
            ),
            "transformacion": (
                "Valor sin modificar antes de aplicar el estado de ruta"
            ),
        },
        {
            "dato": "Capacidad de vehículos por viaje",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "vehículos/viaje",
            "uso_en_modelo": (
                "Restricción del número de vehículos utilizables por ruta"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Estado de ruta post-sismo",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "categoría",
            "uso_en_modelo": (
                "Modificación de disponibilidad, tiempo o capacidad"
            ),
            "transformacion": (
                "Se conserva el texto original; la interpretación "
                "cuantitativa será un parámetro documentado"
            ),
        },
        {
            "dato": "Cantidad de vehículos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "vehículos",
            "uso_en_modelo": (
                "Restricción de la flota disponible por tipo"
            ),
            "transformacion": "Conversión a número entero",
        },
        {
            "dato": "Capacidad de los vehículos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "m³/vehículo",
            "uso_en_modelo": (
                "Capacidad máxima de transporte por viaje"
            ),
            "transformacion": (
                "No se usa para convertir suministros hasta disponer "
                "de equivalencias de litros, raciones y kits a m³"
            ),
        },
        {
            "dato": "Consumo de combustible",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "gal/km",
            "uso_en_modelo": (
                "Cálculo del combustible consumido por viaje"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Combustible disponible",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "galones",
            "uso_en_modelo": (
                "Restricción acumulada de las operaciones logísticas"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Reposiciones externas",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Reposición",
            "unidad": "litros, raciones y kits",
            "uso_en_modelo": (
                "Flujo de entrada al inventario durante la simulación"
            ),
            "transformacion": (
                "Separación de bloque, hora inicial y hora final"
            ),
        },
        {
            "dato": "Población desplazada",
            "fuente": "Pendiente de output formal del Grupo 1",
            "unidad": "personas por zona y bloque",
            "uso_en_modelo": (
                "Conversión de población en demanda de suministros"
            ),
            "transformacion": (
                "personas × consumo_por_persona_bloque"
            ),
        },
        {
            "dato": "Accesibilidad actualizada",
            "fuente": "Pendiente de output formal del Grupo 4",
            "unidad": "estado, horas e índice entre 0 y 1",
            "uso_en_modelo": (
                "Actualización de rutas, tiempos y restricciones de acceso"
            ),
            "transformacion": (
                "Se definirá después del intercambio presencial"
            ),
        },
    ]
)

display(tabla_trazabilidad)

,dato,fuente,unidad,uso_en_modelo,transformacion
0,Inventario inicial de agua,Grupo3_CadenaSuministro.xlsx — Centros de acopio,litros,Estado inicial del inventario de agua por centro,Conversión de formato tabular; valor sin modif...
1,Inventario inicial de alimentos,Grupo3_CadenaSuministro.xlsx — Centros de acopio,raciones,Estado inicial del inventario de alimentos por...,Conversión de formato tabular; valor sin modif...
2,Inventario inicial de medicamentos,Grupo3_CadenaSuministro.xlsx — Centros de acopio,kits,Estado inicial del inventario de medicamentos ...,Conversión de formato tabular; valor sin modif...
3,Capacidad de almacenamiento,Grupo3_CadenaSuministro.xlsx — Centros de acopio,m³,Referencia de capacidad física de cada centro,No se aplica todavía como restricción porque f...
4,Consumo per cápita,Grupo3_CadenaSuministro.xlsx — Tasas de consumo,unidad de suministro/persona/día,"Cálculo de demanda de agua, alimentos y medica...",consumo_diario × PASO_HORAS / 24
5,Distancia de las rutas,Grupo3_CadenaSuministro.xlsx — Red de distribu...,km,Cálculo de tiempo logístico y consumo de combu...,Valor sin modificar; falta definir si se consi...
6,Tiempo normal de ruta,Grupo3_CadenaSuministro.xlsx — Red de distribu...,horas,Programación de llegadas de los vehículos,Valor sin modificar antes de aplicar el estado...
7,Capacidad de vehículos por viaje,Grupo3_CadenaSuministro.xlsx — Red de distribu...,vehículos/viaje,Restricción del número de vehículos utilizable...,Valor sin modificar
8,Estado de ruta post-sismo,Grupo3_CadenaSuministro.xlsx — Red de distribu...,categoría,"Modificación de disponibilidad, tiempo o capac...",Se conserva el texto original; la interpretaci...
9,Cantidad de vehículos,Grupo3_CadenaSuministro.xlsx — Flota,vehículos,Restricción de la flota disponible por tipo,Conversión a número entero


## Descripción del caso

El modelo representa la cadena de suministro de ayuda humanitaria de Ciudad UVG durante las primeras 72 horas posteriores a un terremoto de magnitud 6.8. El sistema debe distribuir agua potable, raciones de alimentos y kits de medicamentos desde cuatro centros de acopio hacia los albergues afectados, usando una red de nueve rutas y una flota heterogénea de camiones, camionetas y motocicletas.

El horizonte se divide en 12 bloques de 6 horas. En cada bloque, los inventarios pueden aumentar por la llegada de reposiciones externas y disminuir por el consumo asociado con la población desplazada. La distribución está limitada por el estado de las rutas, el número de vehículos que puede circular por cada ruta y el combustible disponible.

La unidad principal de análisis es la combinación de zona, suministro y bloque de tiempo. La heterogeneidad del sistema se conserva mediante inventarios iniciales diferentes entre centros, tasas de consumo específicas para cada suministro, rutas con distintas distancias y condiciones, y vehículos con capacidades y consumos de combustible diferentes.

El modelo debe producir tres resultados principales: el balance de suministros por zona, suministro y bloque; la identificación del primer agotamiento crítico sin intervención adicional; y una recomendación de priorización de rutas y frecuencia de distribución que busque maximizar la cobertura bajo la restricción de combustible disponible.

Durante la fase previa al intercambio se utilizan exclusivamente los datos oficiales del Grupo 3. La demanda por zona y bloque se incorporará posteriormente mediante la proyección formal entregada por el Grupo 1, mientras que las condiciones actualizadas de accesibilidad se incorporarán mediante el output formal del Grupo 4. Ambos escenarios deberán conservarse para identificar si la información recibida cambia o confirma la zona inicialmente considerada más crítica.

## Justificación del paradigma

Para seleccionar el paradigma se consideran la naturaleza de las variables, el nivel de agregación, la importancia del orden temporal y el tipo de pregunta que debe responder el modelo. El sistema combina inventarios que evolucionan continuamente por entradas y salidas con operaciones logísticas que ocurren en momentos específicos. Por ello se adopta una arquitectura híbrida basada en **Dinámica de Sistemas (SD)** y **Simulación de Eventos Discretos (DES)**, implementada sobre bloques de 6 horas.

La Dinámica de Sistemas es apropiada para representar los suministros como *stocks*. El inventario de agua, alimentos y medicamentos aumenta mediante las reposiciones recibidas y disminuye mediante el consumo satisfecho. Esta representación permite analizar las trayectorias de inventario, los déficits acumulados y el momento de agotamiento de cada suministro. Aunque la implementación utiliza ecuaciones en tiempo discreto, conserva la lógica fundamental de stocks y flujos.

La Simulación de Eventos Discretos complementa este componente porque las salidas y llegadas de convoyes, los viajes por las rutas y las reposiciones externas ocurren como eventos identificables. Estos eventos modifican el estado del sistema en momentos determinados y están condicionados por el tiempo de viaje, la capacidad de las rutas, la disponibilidad de vehículos y el combustible restante. Su representación permite distinguir entre inventario almacenado, inventario en tránsito e inventario efectivamente recibido.

La frontera entre ambos componentes se define de la siguiente manera: SD representa el balance agregado de inventarios y demanda por zona, suministro y bloque, mientras que DES representa las operaciones logísticas que trasladan suministros y determinan cuándo una carga pasa a estar disponible en su destino. El estado resultante de los eventos logísticos alimenta las entradas del balance de inventario.

Un modelo exclusivamente de Dinámica de Sistemas sería suficiente para estudiar un inventario agregado si las entregas fueran instantáneas y las rutas no importaran. Sin embargo, representaría con menor precisión los retrasos, bloqueos y llegadas de convoyes que afectan la disponibilidad real de la ayuda. Por otra parte, un modelo exclusivamente DES podría representar cada viaje, pero complicaría innecesariamente el cálculo agregado del consumo y del balance de suministros.

No se selecciona un Modelo Basado en Agentes (ABM) porque los datos disponibles no describen decisiones autónomas, aprendizaje o interacción social entre vehículos, centros o albergues. Los vehículos siguen reglas logísticas centralizadas y no poseen objetivos individuales. Incluir agentes no aportaría capacidad explicativa para justificar el aumento de complejidad.

## Entidades y variables de estado

El modelo representa la cadena de suministro mediante centros de acopio, destinos de ayuda, tipos de suministro, rutas de distribución, vehículos y reposiciones externas. La heterogenedad del sistema se conserva porque cada entidad posee atributos diferentes y no todas las zonas tienen el mismo inventario, acceso o capacidad logística.

### Centros de acopio

Los centros almacenan los suministros disponibles al inicio y reciben, reposiciones externas. El Excel incluye cuatro centros:

- Bodega Central CONRED, ubicada en Z4.
- Almacén Norte, ubicado en Z2.
- Depósito Sur, ubicado en Z3.
- Centro Logístico Z5, ubicado en Z5.

Sus atributos son:

- `centro`: nombre del centro.
- `zona`: identificador de la zona donde está ubicado.
- `agua_litros`: inventario inicial de agua.
- `alimentos_raciones`: inventario inicial de alimentos.
- `medicamentos_kits`: inventario inicial de medicamentos.
- `capacidad_almacenamiento_m3`: capacidad física de almacenamiento.

### Destinos de ayuda

Los destinos representan los albergues que reciben suministros. El Excel identifica los siguientes destinos:

- Albergue Estadio.
- Albergue Z2.
- Albergue Z3.
- Albergue Z5-1.
- Albergue Z5-2.

Cada destino se caracteriza por:

- `destino`: nombre del albergue.
- `zona`: zona urbana correspondiente.
- `demanda`: cantidad solicitada durante el bloque.
- `consumo_satisfecho`: parte de la demanda atendida.
- `demanda_no_satisfecha`: parte de la demanda no atendida.
- `inventario_disponible`: suministros disponibles en el destino.

La zona del Albergue Estadio no aparece explícitamente en el Excel. Por tanto, su asignación debe permanecer registrada como dato pendiente o como supuesto explícito.

### Tipos de suministro

El modelo distingue tres suministros porque tienen unidades, inventarios y tasas de consumo diferentes:

- Agua potable, medida en litros.
- Alimentos, medidos en raciones.
- Medicamentos básicos, medidos en kits.

Para cada suministro se registra:

- `suministro`: identificador normalizado.
- `consumo_por_persona_dia`: tasa original proporcionada por el Excel.
- `consumo_por_persona_bloque`: tasa convertida a un bloque de 6 horas.
- `unidad`: unidad física correspondiente.

### Rutas de distribución

Las rutas conectan centros de acopio con albergues. Cada ruta contiene:

- `ruta_id`: identificador único.
- `origen`: centro de acopio de salida.
- `destino`: albergue receptor.
- `distancia_km`: distancia del trayecto.
- `tiempo_normal_h`: duración esperada sin afectaciones.
- `capacidad_vehiculos_viaje`: máximo de vehículos permitidos por viaje.
- `estado_ruta`: condición posterior al terremoto.

### Vehículos

La flota contiene camiones grandes, camionetas y motocicletas. Los tipos se mantienen separados porque difieren en capacidad, cantidad disponible y consumo de combustible.

Los atributos son:

- `tipo_vehiculo`: identificador normalizado.
- `cantidad_disponible`: número de unidades operativas.
- `capacidad_m3`: volumen máximo transportable por vehículo.
- `consumo_combustible_gal_km`: galones consumidos por kilómetro.
- `combustible_disponible_gal`: reserva disponible para ese tipo.

Las motocicletas se conservan dentro de la estructura de la flota, aunque su función principal sea mensajería. 

### Reposiciones externas

Las reposiciones representan las donaciones o llegadas externas programadas. Sus atributos son:

- `bloque`: bloque en que la reposición se vuelve disponible.
- `hora_inicio`: inicio del bloque.
- `hora_fin`: final del bloque.
- `agua_adicional_litros`: agua recibida.
- `alimentos_adicionales_raciones`: alimentos recibidos.
- `medicamentos_adicionales_kits`: medicamentos recibidos.
- `fuente`: entidad responsable de la reposición.

### Variables de estado

Para cada combinación de bloque, zona y suministro, el modelo registra:

- `tiempo_h`: tiempo transcurrido desde el terremoto.
- `bloque`: período de 6 horas.
- `zona`: zona atendida.
- `centro`: centro responsable del inventario.
- `suministro`: agua, alimentos o medicamentos.
- `inventario_inicio`: cantidad disponible al comenzar el bloque.
- `demanda`: cantidad requerida durante el bloque.
- `consumo_satisfecho`: demanda que puede cubrirse.
- `demanda_no_satisfecha`: demanda que no puede cubrirse.
- `reposicion_recibida`: entrada disponible en el bloque.
- `inventario_fin`: cantidad restante.
- `superavit`: disponibilidad restante después de atender la demanda.
- `deficit`: demanda que permanece sin atender.

Cuando se modelan las operaciones logísticas, también se registran:

- `ruta_id`: ruta utilizada.
- `tipo_vehiculo`: vehículo asignado.
- `viajes`: número de viajes realizados.
- `carga_transportada`: cantidad trasladada.
- `combustible_consumido`: combustible utilizado.
- `combustible_restante`: combustible disponible después del viaje.
- `tiempo_transito`: duración efectiva del traslado.
- `inventario_en_transito`: suministros enviados pero todavía no recibidos.

### Escala temporal y heterogeneidad

El horizonte es de 72 horas, dividido en 12 bloques de 6 horas. El estado se actualiza una vez por bloque, pero los eventos logísticos conservan su tiempo de llegada para impedir que una carga se utilice antes de arribar.

La heterogeneidad se representa mediante inventarios iniciales diferentes entre centros, tasas específicas por suministro, capacidades distintas entre vehículos y rutas con diferentes distancias, tiempos, capacidades y estados posteriores al terremoto.

## Supuestos del modelo

Los siguientes supuestos se utilizan para resolver ambigüedades del Excel y delimitar qué puede modelarse antes del intercambio presencial. Se distinguen de los datos originales para evitar presentar decisiones del grupo como si fueran observaciones proporcionadas.

### Convención temporal

Se adopta una numeración de bloques iniciada en cero. Por tanto:

- T0 corresponde a 0–6 horas.
- T1 corresponde a 6–12 horas.
- T2 corresponde a 12–18 horas.
- T11 corresponde a 66–72 horas.

Esta convención coincide con las etiquetas temporales de las reposiciones del Excel. Una reposición asociada con un bloque se procesa al inicio de ese bloque, antes de atender su demanda, siempre que ya haya llegado al centro correspondiente.

### Demanda de suministros

El Excel proporciona tasas de consumo por persona, pero no contiene la cantidad de personas desplazadas por zona y bloque. La demanda se calculará posteriormente mediante:

\[
D_{z,s,t}=P_{z,t}\,c_{s},
\]

donde \(P_{z,t}\) es la población que requiere suministros en la zona \(z\) durante el bloque \(t\), y \(c_s\) es el consumo por persona y bloque del suministro \(s\).

Antes del intercambio no se inventarán valores de población para producir conclusiones sobre agotamiento. El motor podrá probarse con datos controlados exclusivamente para verificar su consistencia, pero estos resultados no se presentarán como resultados sustantivos del modelo. La demanda formal será incorporada a partir del output recibido del Grupo 1.

### Correspondencia entre centros y nombres de rutas

El Excel utiliza nombres diferentes para dos centros:

| Nombre en centros | Nombre en rutas |
|---|---|
| Bodega Central CONRED | Bodega Central |
| Centro Logístico Z5 | Centro Log. Z5 |

Se asumirá que cada par representa la misma instalación. La correspondencia se implementará mediante un mapeo explícito y configurable; no se reemplazarán silenciosamente los nombres originales.

### Zona del Albergue Estadio

El Excel no especifica la zona urbana del Albergue Estadio. Por tanto, no se le asignará una zona definitiva hasta contar con una aclaración del catedrático o con el output formal recibido durante el intercambio. Mientras tanto, el destino conservará el identificador `Albergue Estadio`.

### Cobertura de Z1

La red inicial no presenta un centro o una ruta cuyo nombre identifique explícitamente a Z1. En consecuencia, el modelo no supondrá una ruta inexistente. Si el Grupo 1 reporta demanda en Z1, esta quedará identificada como demanda sin una ruta inicial de abastecimiento hasta incorporar el mapa formal del Grupo 4.

### Interpretación de “velocidad reducida 60 %”

La frase “velocidad reducida 60 %” es ambigua: puede significar que la velocidad disminuye en 60 % o que el vehículo circula al 60 % de la velocidad normal. Se adoptará como escenario base la interpretación conservadora de que la velocidad disminuye 60 %, por lo que queda en 40 % de su valor normal.

Esta interpretación se almacenará como parámetro configurable y deberá evaluarse mediante análisis de sensibilidad frente a la interpretación alternativa.

### Rutas parcialmente bloqueadas

El Excel no especifica cuánto disminuye la capacidad ni cuánto aumenta el tiempo de las rutas parcialmente bloqueadas. Antes del intercambio se conservará su estado como categoría, pero no se inventará una reducción porcentual.

Estas rutas no se utilizarán para una recomendación logística definitiva hasta recibir las capacidades o condiciones actualizadas del Grupo 4. Si se requiere una ejecución técnica del motor, la política provisional utilizada deberá identificarse expresamente como prueba y no como resultado del análisis.

### Distancia y retorno de vehículos

La distancia indicada en el Excel se interpreta como distancia de ida entre el centro y el albergue. Para calcular el combustible necesario para que un vehículo complete su operación y regrese al centro, se utiliza una distancia de ida y vuelta:

\[
d_{\text{recorrida}}=2d_{\text{ruta}}.
\]

En consecuencia:

\[
G_{r,v}
=
2d_r c_v,
\]

donde \(G_{r,v}\) es el combustible utilizado por un vehículo de tipo \(v\) en la ruta \(r\), \(d_r\) es la distancia de ida y \(c_v\) es el consumo en galones por kilómetro.

Si el vehículo no necesita regresar dentro del horizonte analizado, este factor podrá modificarse mediante un parámetro configurable.

### Asignación de reposiciones externas

El Excel especifica cantidades y momentos de reposición, pero no indica el centro receptor. Por ello, las reposiciones no se sumarán automáticamente a todos los centros, ya que esto duplicaría los suministros.

La política de recepción permanecerá configurable. Como escenario provisional, y únicamente si se necesita ejecutar el modelo antes de obtener una aclaración, se asignarán a la Bodega Central CONRED por ser el centro identificado como bodega central y poseer el mayor inventario inicial. Esta decisión se reportará como supuesto y deberá someterse a análisis de sensibilidad.

### Capacidad expresada en metros cúbicos

El Excel expresa en metros cúbicos la capacidad de los centros y vehículos, pero no proporciona factores para convertir litros de agua, raciones de alimentos y kits de medicamentos a volumen transportado.

Por esta razón, las capacidades en metros cúbicos se conservarán y reportarán, pero no se utilizarán inicialmente como restricciones cuantitativas. Aplicarlas sin equivalencias produciría una comparación entre unidades incompatibles. El modelo podrá incorporar esta restricción cuando se disponga de factores de volumen por suministro.

### Inventario en centros y recepción en destinos

El inventario inicial pertenece a los centros de acopio. Los albergues no reciben inventario inicial porque el Excel no lo proporciona. Los suministros enviados se registrarán como inventario en tránsito y solo podrán atender demanda cuando el evento de llegada haya ocurrido.

### Motocicletas

Las motocicletas se conservarán como parte de la flota, pero no se supondrá que transportan agua, alimentos o medicamentos en las mismas condiciones que los vehículos de carga. Su función original se identifica como mensajería. No se les asignará carga humanitaria sin una regla o equivalencia explícita.

### Incertidumbre

No se asignarán distribuciones probabilísticas arbitrarias. La incertidumbre se incorporará siguiendo este orden:

1. Distribuciones o rangos proporcionados por los datos.
2. Variabilidad calculable a partir de los outputs recibidos.
3. Supuestos explícitos, conservadores y sometidos a sensibilidad.

Todos los parámetros estocásticos permanecerán centralizados en la configuración del modelo y no se ocultarán dentro de las funciones.

## Ecuaciones y reglas

### Índices del modelo

Se utilizan los siguientes índices:

- $c$: centro de acopio.
- $z$: zona o destino.
- $s$: tipo de suministro.
- $r$: ruta de distribución.
- $v$: tipo de vehículo.
- $t$: bloque de tiempo, con $t=0,1,\ldots,11$.

Cada bloque tiene una duración de:

$$
\Delta t=6\ \text{horas}.
$$

### Demanda por zona y suministro

La demanda de cada suministro se obtiene multiplicando la población que requiere ayuda por la tasa de consumo correspondiente:

$$
D_{z,s,t}=P_{z,t}\,c_s.
$$

Donde:

- $D_{z,s,t}$ es la demanda del suministro $s$ en la zona $z$ durante el bloque $t$.
- $P_{z,t}$ es la cantidad de personas que requieren suministros.
- $c_s$ es el consumo por persona y bloque del suministro $s$.

Las tasas utilizadas se calculan desde los datos del Excel:

$$
c_{\text{agua}}
=
3\left(\frac{6}{24}\right)
=
0.75\ \text{litros/persona/bloque}.
$$

$$
c_{\text{alimentos}}
=
1\left(\frac{6}{24}\right)
=
0.25\ \text{raciones/persona/bloque}.
$$

$$
c_{\text{medicamentos}}
=
0.05\left(\frac{6}{24}\right)
=
0.0125\ \text{kits/persona/bloque}.
$$

### Balance de inventario en los centros

El inventario de un centro aumenta mediante las reposiciones externas y disminuye cuando se despachan suministros:

$$
I^{C,\text{fin}}_{c,s,t}
=
I^{C,\text{inicio}}_{c,s,t}
+
R_{c,s,t}
-
X_{c,s,t}.
$$

Donde:

- $I^{C,\text{inicio}}_{c,s,t}$ es el inventario del centro al inicio del bloque.
- $R_{c,s,t}$ es la reposición externa recibida.
- $X_{c,s,t}$ es la cantidad despachada hacia los destinos.
- $I^{C,\text{fin}}_{c,s,t}$ es el inventario restante al final del bloque.

La cantidad despachada no puede superar la disponibilidad del centro:

$$
0
\leq
X_{c,s,t}
\leq
I^{C,\text{inicio}}_{c,s,t}
+
R_{c,s,t}.
$$

El inventario final de un bloque se convierte en el inventario inicial del siguiente:

$$
I^{C,\text{inicio}}_{c,s,t+1}
=
I^{C,\text{fin}}_{c,s,t}.
$$

### Inventario en tránsito

Una carga despachada no puede utilizarse inmediatamente. Mientras el vehículo se encuentra viajando, la cantidad se registra como inventario en tránsito.

Para una ruta $r$, el número de bloques necesarios para completar el traslado se calcula como:

$$
L_{r,t}
=
\max\left(
1,
\left\lceil
\frac{\tau_{r,t}}{\Delta t}
\right\rceil
\right).
$$

Donde:

- $L_{r,t}$ es el número de bloques requerido para completar la ruta.
- $\tau_{r,t}$ es el tiempo efectivo de viaje.
- $\Delta t$ es la duración de un bloque.

Si una carga sale en el bloque $t$, su bloque de llegada es:

$$
t_{\text{llegada}}
=
t+L_{r,t}.
$$

La cantidad que llega a una zona durante un bloque es la suma de las cargas cuyo evento de llegada corresponde a ese momento:

$$
A_{z,s,t}
=
\sum_{\substack{
r:\,\operatorname{destino}(r)=z
}}
X_{r,s,t-L_{r,t}}.
$$

### Disponibilidad en los destinos

La disponibilidad antes del consumo se calcula como:

$$
B_{z,s,t}
=
I^{Z,\text{inicio}}_{z,s,t}
+
A_{z,s,t}.
$$

Donde:

- $I^{Z,\text{inicio}}_{z,s,t}$ es el inventario existente en el destino al inicio del bloque.
- $A_{z,s,t}$ es la cantidad recibida mediante los eventos de llegada.
- $B_{z,s,t}$ es la disponibilidad total antes de atender la demanda.

### Consumo satisfecho

La cantidad de demanda atendida corresponde al mínimo entre la disponibilidad y la demanda:

$$
C_{z,s,t}
=
\min\left(
B_{z,s,t},
D_{z,s,t}
\right).
$$

Esta regla impide entregar más suministros de los disponibles o más de lo solicitado.

### Demanda no satisfecha y déficit

La demanda no satisfecha se calcula explícitamente como:

$$
NS_{z,s,t}
=
\max\left(
0,
D_{z,s,t}-B_{z,s,t}
\right).
$$

El déficit del bloque se define como:

$$
F_{z,s,t}=NS_{z,s,t}.
$$

No se permite ocultar el déficit limitando únicamente el inventario a cero. La demanda no satisfecha debe almacenarse como una variable independiente.

En el modelo base, la demanda no satisfecha se reporta en el bloque donde ocurre, pero no se transfiere automáticamente al siguiente bloque. Si posteriormente se decide acumularla, esa modificación deberá documentarse como una nueva regla.

### Superávit e inventario final en los destinos

El superávit después de atender la demanda es:

$$
S_{z,s,t}
=
\max\left(
0,
B_{z,s,t}-D_{z,s,t}
\right).
$$

El inventario final del destino es equivalente al superávit:

$$
I^{Z,\text{fin}}_{z,s,t}
=
B_{z,s,t}
-
C_{z,s,t}
=
S_{z,s,t}.
$$

El inventario final de un bloque se convierte en el inventario inicial del siguiente:

$$
I^{Z,\text{inicio}}_{z,s,t+1}
=
I^{Z,\text{fin}}_{z,s,t}.
$$

### Primer agotamiento crítico

Se considera que ocurre un agotamiento crítico cuando la disponibilidad no alcanza para cubrir la demanda:

$$
I^{Z,\text{fin}}_{z,s,t}=0
\quad\land\quad
NS_{z,s,t}>0.
$$

El primer agotamiento corresponde a la combinación $(z,s,t)$ con el menor bloque que satisface ambas condiciones. Esta definición distingue entre terminar exactamente con inventario cero y experimentar una escasez efectiva.

### Tiempo efectivo de las rutas

Para una ruta disponible:

$$
\tau_{r,t}
=
\tau^{\text{normal}}_r.
$$

Para una ruta cuya velocidad disminuye en 60 %, la velocidad efectiva corresponde al 40 % de la normal. Por tanto:

$$
\tau_{r,t}
=
\frac{\tau^{\text{normal}}_r}{0.40}.
$$

La interpretación de las rutas parcialmente bloqueadas permanecerá como un parámetro pendiente hasta recibir información cuantitativa del Grupo 4.

### Combustible por viaje

Si la distancia proporcionada por el Excel corresponde al trayecto de ida y se requiere el retorno del vehículo, la distancia total recorrida es:

$$
d^{\text{recorrida}}_r
=
2d_r.
$$

El combustible utilizado por un vehículo de tipo $v$ en la ruta $r$ se calcula como:

$$
G_{r,v}
=
2d_r g_v.
$$

Donde:

- $G_{r,v}$ es el combustible consumido por viaje.
- $d_r$ es la distancia de ida.
- $g_v$ es el consumo del vehículo en galones por kilómetro.

Si se asignan $n_{r,v,t}$ vehículos, el consumo total es:

$$
G^{\text{total}}_{r,v,t}
=
n_{r,v,t}\,2d_r g_v.
$$

### Restricciones de vehículos y combustible

El número de vehículos asignados a una ruta no puede superar su capacidad vehicular:

$$
\sum_v n_{r,v,t}
\leq
K_r,
$$

donde $K_r$ es la cantidad máxima de vehículos permitidos por viaje en la ruta.

La cantidad asignada tampoco puede superar la flota disponible:

$$
\sum_r n_{r,v,t}
\leq
N_{v,t},
$$

donde $N_{v,t}$ es la cantidad disponible del tipo de vehículo $v$.

El combustible consumido no puede superar el combustible restante:

$$
\sum_r G^{\text{total}}_{r,v,t}
\leq
G^{\text{disponible}}_{v,t}.
$$

El combustible al final del bloque se actualiza mediante:

$$
G^{\text{fin}}_{v,t}
=
G^{\text{inicio}}_{v,t}
-
\sum_r G^{\text{total}}_{r,v,t}.
$$

El combustible final se convierte en el combustible inicial del siguiente bloque:

$$
G^{\text{inicio}}_{v,t+1}
=
G^{\text{fin}}_{v,t}.
$$

### Cobertura de la demanda

La cobertura por zona, suministro y bloque se calcula como:

$$
\operatorname{Cobertura}_{z,s,t}
=
\begin{cases}
\dfrac{C_{z,s,t}}{D_{z,s,t}},
& \text{si } D_{z,s,t}>0,\\
1,
& \text{si } D_{z,s,t}=0.
\end{cases}
$$

La cobertura total de un escenario se calcula como:

$$
\operatorname{Cobertura\ total}
=
\frac{
\sum_{z,s,t} C_{z,s,t}
}{
\sum_{z,s,t} D_{z,s,t}
}.
$$

Esta métrica permitirá comparar políticas de priorización de rutas y frecuencia de distribución bajo las restricciones de acceso, disponibilidad de vehículos y combustible.

## Scheduling

El modelo utiliza un scheduling híbrido y sincrónico organizado en bloques de 6 horas. Los balances de inventario se actualizan una vez por bloque, mientras que las salidas, llegadas y retornos de vehículos se representan como eventos discretos programados.

El orden de actualización permanece constante durante los 12 bloques, ya que modificarlo podría cambiar la disponibilidad de suministros y el momento de agotamiento.

### Convención de los bloques

La simulación utiliza bloques numerados desde T0 hasta T11:

| Bloque | Intervalo |
|---:|---:|
| T0 | 0–6 h |
| T1 | 6–12 h |
| T2 | 12–18 h |
| T3 | 18–24 h |
| T4 | 24–30 h |
| T5 | 30–36 h |
| T6 | 36–42 h |
| T7 | 42–48 h |
| T8 | 48–54 h |
| T9 | 54–60 h |
| T10 | 60–66 h |
| T11 | 66–72 h |

El estado observado al comienzo de un bloque corresponde al estado final del bloque anterior.

### Orden de actualización dentro de cada bloque

En cada bloque $t$ se ejecutan las siguientes operaciones:

1. **Observar el estado inicial.**  
   Se registran los inventarios de los centros y destinos, el inventario en tránsito, el combustible restante y los vehículos disponibles.

2. **Procesar los retornos de vehículos.**  
   Los vehículos cuyo evento de retorno ocurre en el bloque actual vuelven a estar disponibles. Un vehículo que todavía se encuentra en ruta no puede asignarse a otro viaje.

3. **Procesar las reposiciones externas.**  
   Las reposiciones programadas para el bloque actual ingresan al centro receptor antes de planificar nuevos despachos. Por ejemplo, la reposición T3 se vuelve disponible al inicio del intervalo de 18 a 24 horas.

4. **Procesar las llegadas a los destinos.**  
   Las cargas cuyo evento de llegada corresponde al bloque actual dejan de formar parte del inventario en tránsito y se agregan al inventario del destino.

5. **Calcular la demanda del bloque.**  
   La población correspondiente a cada zona se multiplica por las tasas de consumo por persona y bloque:

   $$
   D_{z,s,t}=P_{z,t}c_s.
   $$

6. **Atender la demanda.**  
   Para cada zona y suministro se calcula el consumo satisfecho como el mínimo entre la disponibilidad y la demanda:

   $$
   C_{z,s,t}
   =
   \min(B_{z,s,t},D_{z,s,t}).
   $$

7. **Registrar déficit y superávit.**  
   Se calcula y almacena explícitamente la demanda no satisfecha:

   $$
   NS_{z,s,t}
   =
   \max(0,D_{z,s,t}-B_{z,s,t}).
   $$

   También se registra el inventario restante o superávit:

   $$
   S_{z,s,t}
   =
   \max(0,B_{z,s,t}-D_{z,s,t}).
   $$

8. **Determinar las necesidades de distribución.**  
   Con base en el déficit observado, la demanda proyectada del siguiente bloque y los inventarios disponibles, la política logística determina qué zonas y suministros requieren prioridad.

9. **Seleccionar rutas y vehículos.**  
   Antes de autorizar un despacho se verifica:

   - Que la ruta pueda utilizarse.
   - Que el centro tenga inventario suficiente.
   - Que existan vehículos disponibles.
   - Que no se exceda la capacidad vehicular de la ruta.
   - Que exista combustible suficiente para el recorrido.
   - Que la carga pueda transportarse con las unidades disponibles, cuando existan factores de conversión compatibles.

10. **Despachar y programar eventos.**  
    La carga autorizada se resta del centro y se registra como inventario en tránsito. También se descuenta el combustible y se programan por separado:

    - El evento de llegada de la carga al destino.
    - El evento de retorno del vehículo al centro.

11. **Guardar el estado final.**  
    Se registran los inventarios finales, el combustible restante, los vehículos ocupados, las cargas en tránsito, el consumo satisfecho y la demanda no satisfecha.

12. **Transferir el estado al bloque siguiente.**  
    El estado final del bloque $t$ se convierte en el estado inicial del bloque $t+1$.

### Regla de sincronización

La demanda y la disponibilidad de todas las zonas se calculan antes de modificar los inventarios por nuevas decisiones de despacho. Esto evita que el resultado dependa accidentalmente del orden de las filas del DataFrame.

Cuando varias zonas compiten por un mismo inventario, la asignación se realiza mediante una política de prioridad explícita. No se permite que la primera zona almacenada en la tabla reciba ayuda automáticamente solo por aparecer primero.

### Regla para llegadas dentro de un bloque

Debido a que el modelo actualiza sus balances en intervalos de 6 horas, todo viaje con tiempo positivo requiere al menos un bloque para producir una entrega disponible:

$$
L_{r,t}
=
\max\left(
1,
\left\lceil
\frac{\tau_{r,t}}{6}
\right\rceil
\right).
$$

Por tanto, una carga despachada durante T2 no puede satisfacer demanda de T2. Si requiere un bloque de traslado, estará disponible al inicio de T3.

Esta convención es conservadora y evita utilizar suministros antes de que ocurra su llegada. Como limitación, no representa con detalle entregas que salen y llegan dentro del mismo intervalo de 6 horas.

### Eventos fuera del horizonte

Los despachos realizados al final de la simulación pueden tener una llegada o retorno posterior a las 72 horas. Estos eventos se conservan como pendientes, pero sus cargas no se contabilizan como suministros entregados dentro del horizonte analizado.

### Reproducibilidad

Cada realización utiliza el mismo orden de actualización y una semilla identificable. La semilla controla únicamente los componentes estocásticos; no modifica las reglas de scheduling. Una realización ejecutada nuevamente con la misma semilla debe producir exactamente la misma trayectoria.

## Motor de simulación

In [14]:
def crear_tabla_demanda(
    poblacion: pd.DataFrame,
    consumo: pd.DataFrame,
    configuracion: dict[str, Any],
    rng: np.random.Generator | None = None,
) -> pd.DataFrame:

    columnas_salida = [
        "bloque",
        "tiempo_h",
        "zona",
        "personas",
        "suministro",
        "consumo_por_persona_bloque",
        "demanda",
        "unidad",
    ]

    if poblacion.empty:
        return pd.DataFrame(columns=columnas_salida)

    columnas_poblacion = {
        "bloque",
        "zona",
        "personas",
    }

    columnas_consumo = {
        "suministro",
        "consumo_por_persona_bloque",
        "unidad_por_bloque",
    }

    faltantes_poblacion = columnas_poblacion.difference(
        poblacion.columns
    )

    faltantes_consumo = columnas_consumo.difference(
        consumo.columns
    )

    if faltantes_poblacion:
        raise ValueError(
            "La tabla de población no contiene las columnas: "
            f"{sorted(faltantes_poblacion)}"
        )

    if faltantes_consumo:
        raise ValueError(
            "La tabla de consumo no contiene las columnas: "
            f"{sorted(faltantes_consumo)}"
        )

    poblacion_validada = poblacion[
        ["bloque", "zona", "personas"]
    ].copy()

    poblacion_validada["bloque"] = pd.to_numeric(
        poblacion_validada["bloque"],
        errors="raise",
    ).astype(int)

    poblacion_validada["personas"] = pd.to_numeric(
        poblacion_validada["personas"],
        errors="raise",
    )

    if poblacion_validada[
        ["bloque", "zona", "personas"]
    ].isna().any().any():
        raise ValueError(
            "La tabla de población contiene valores faltantes."
        )

    if (poblacion_validada["personas"] < 0).any():
        raise ValueError(
            "La cantidad de personas no puede ser negativa."
        )

    if poblacion_validada.duplicated(
        subset=["bloque", "zona"]
    ).any():
        duplicados = poblacion_validada.loc[
            poblacion_validada.duplicated(
                subset=["bloque", "zona"],
                keep=False,
            ),
            ["bloque", "zona"],
        ]

        raise ValueError(
            "Existen combinaciones duplicadas de bloque y zona:\n"
            f"{duplicados.to_string(index=False)}"
        )

    bloques_esperados = set(
        range(configuracion["n_pasos"])
    )

    bloques_observados = set(
        poblacion_validada["bloque"]
    )

    if bloques_observados != bloques_esperados:
        faltantes = sorted(
            bloques_esperados - bloques_observados
        )
        adicionales = sorted(
            bloques_observados - bloques_esperados
        )

        raise ValueError(
            "Los bloques de población no coinciden con el "
            "horizonte del modelo. "
            f"Faltantes: {faltantes}. "
            f"Fuera del horizonte: {adicionales}."
        )

    zonas_esperadas = set(
        configuracion["zonas_esperadas"]
    )

    zonas_observadas = set(
        poblacion_validada["zona"]
    )

    if zonas_observadas != zonas_esperadas:
        faltantes = sorted(
            zonas_esperadas - zonas_observadas
        )
        adicionales = sorted(
            zonas_observadas - zonas_esperadas
        )

        raise ValueError(
            "Las zonas de población no coinciden con las "
            "cinco zonas del modelo. "
            f"Faltantes: {faltantes}. "
            f"No reconocidas: {adicionales}."
        )

    combinaciones_esperadas = pd.MultiIndex.from_product(
        [
            range(configuracion["n_pasos"]),
            configuracion["zonas_esperadas"],
        ],
        names=["bloque", "zona"],
    )

    combinaciones_observadas = pd.MultiIndex.from_frame(
        poblacion_validada[["bloque", "zona"]]
    )

    combinaciones_faltantes = (
        combinaciones_esperadas.difference(
            combinaciones_observadas
        )
    )

    if len(combinaciones_faltantes) > 0:
        raise ValueError(
            "Faltan combinaciones de bloque y zona: "
            f"{combinaciones_faltantes.tolist()}"
        )

    poblacion_validada["tiempo_h"] = (
        poblacion_validada["bloque"]
        * configuracion["paso_horas"]
    )

    demanda = poblacion_validada.merge(
        consumo[
            [
                "suministro",
                "consumo_por_persona_bloque",
                "unidad_por_bloque",
            ]
        ],
        how="cross",
    )

    cv_demanda = configuracion[
        "incertidumbre"
    ]["cv_demanda"]

    if cv_demanda < 0:
        raise ValueError(
            "cv_demanda no puede ser negativo."
        )

    if cv_demanda > 0:
        if rng is None:
            raise ValueError(
                "Se necesita un generador rng cuando "
                "cv_demanda es mayor que cero."
            )

        sigma_log = np.sqrt(
            np.log(1 + cv_demanda**2)
        )
        media_log = -0.5 * sigma_log**2

        multiplicador = rng.lognormal(
            mean=media_log,
            sigma=sigma_log,
            size=len(demanda),
        )
    else:
        multiplicador = np.ones(len(demanda))

    demanda["demanda"] = (
        demanda["personas"]
        * demanda["consumo_por_persona_bloque"]
        * multiplicador
    )

    demanda = demanda.rename(
        columns={
            "unidad_por_bloque": "unidad",
        }
    )

    return (
        demanda[columnas_salida]
        .sort_values(
            ["bloque", "zona", "suministro"]
        )
        .reset_index(drop=True)
    )

poblacion_pendiente = pd.DataFrame(
    columns=[
        "bloque",
        "zona",
        "personas",
    ]
)

df_demanda = crear_tabla_demanda(
    poblacion=poblacion_pendiente,
    consumo=datos_modelo["consumo"],
    configuracion=configuracion_modelo,
)

datos_modelo["demanda"] = df_demanda

print("MOTOR DE DEMANDA: PASA")
print("Demanda formal pendiente del Grupo 1.")
print("Filas actuales:", len(datos_modelo["demanda"]))

display(datos_modelo["demanda"])

MOTOR DE DEMANDA: PASA
Demanda formal pendiente del Grupo 1.
Filas actuales: 0


,bloque,tiempo_h,zona,personas,suministro,consumo_por_persona_bloque,demanda,unidad


### Lógica de rutas dañadas

In [15]:
def preparar_rutas_simulacion(
    rutas: pd.DataFrame,
    configuracion: dict[str, Any],
    rutas_actualizadas: pd.DataFrame | None = None,
) -> pd.DataFrame:

    columnas_requeridas = {
        "ruta_id",
        "ruta_original",
        "origen",
        "destino",
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
        "estado_ruta",
    }

    faltantes = columnas_requeridas.difference(
        rutas.columns
    )

    if faltantes:
        raise ValueError(
            "La tabla de rutas no contiene las columnas: "
            f"{sorted(faltantes)}"
        )

    rutas_base = rutas.copy()

    rutas_base = rutas_base.rename(
        columns={
            "origen": "origen_original",
            "estado_ruta": "estado_ruta_original",
            "capacidad_vehiculos_viaje": (
                "capacidad_vehiculos_normal"
            ),
        }
    )

    rutas_base["centro"] = rutas_base[
        "origen_original"
    ].map(
        configuracion["mapeo_nombres_centros"]
    )

    rutas_base["zona"] = rutas_base[
        "destino"
    ].map(
        configuracion["mapeo_destinos_zonas"]
    )

    if rutas_base["centro"].isna().any():
        origenes_sin_mapeo = rutas_base.loc[
            rutas_base["centro"].isna(),
            "origen_original",
        ].unique().tolist()

        raise ValueError(
            "Existen nombres de centros sin correspondencia: "
            f"{origenes_sin_mapeo}"
        )

    bloques = pd.DataFrame(
        {
            "bloque": range(
                configuracion["n_pasos"]
            )
        }
    )

    rutas_bloque = rutas_base.merge(
        bloques,
        how="cross",
    )

    rutas_bloque["tiempo_h"] = (
        rutas_bloque["bloque"]
        * configuracion["paso_horas"]
    )

    rutas_bloque["ruta_disponible"] = False
    rutas_bloque["tiempo_transito_h"] = np.nan
    rutas_bloque["capacidad_vehiculos_viaje"] = 0
    rutas_bloque["fuente_estado_ruta"] = (
        "Datos iniciales del Grupo 3"
    )

    estado_disponible = (
        rutas_bloque["estado_ruta_original"]
        == "Disponible"
    )

    rutas_bloque.loc[
        estado_disponible,
        "ruta_disponible",
    ] = True

    rutas_bloque.loc[
        estado_disponible,
        "tiempo_transito_h",
    ] = rutas_bloque.loc[
        estado_disponible,
        "tiempo_normal_h",
    ]

    rutas_bloque.loc[
        estado_disponible,
        "capacidad_vehiculos_viaje",
    ] = rutas_bloque.loc[
        estado_disponible,
        "capacidad_vehiculos_normal",
    ]

    estado_danado = (
        rutas_bloque["estado_ruta_original"]
        == "Dañada — velocidad reducida 60%"
    )

    fraccion_velocidad = configuracion[
        "fraccion_velocidad_ruta_danada"
    ]

    rutas_bloque.loc[
        estado_danado,
        "ruta_disponible",
    ] = True

    rutas_bloque.loc[
        estado_danado,
        "tiempo_transito_h",
    ] = (
        rutas_bloque.loc[
            estado_danado,
            "tiempo_normal_h",
        ]
        / fraccion_velocidad
    )

    rutas_bloque.loc[
        estado_danado,
        "capacidad_vehiculos_viaje",
    ] = rutas_bloque.loc[
        estado_danado,
        "capacidad_vehiculos_normal",
    ]

    estado_parcial = (
        rutas_bloque["estado_ruta_original"]
        == "Parcialmente bloqueada"
    )

    permitir_parciales = configuracion[
        "permitir_rutas_parciales_sin_actualizacion"
    ]

    if permitir_parciales:
        factor_tiempo = configuracion[
            "factor_tiempo_ruta_parcial"
        ]
        factor_capacidad = configuracion[
            "factor_capacidad_ruta_parcial"
        ]

        if (
            factor_tiempo is None
            or factor_capacidad is None
        ):
            raise ValueError(
                "Para habilitar rutas parcialmente bloqueadas "
                "se requieren factores explícitos de tiempo y "
                "capacidad."
            )

        rutas_bloque.loc[
            estado_parcial,
            "ruta_disponible",
        ] = True

        rutas_bloque.loc[
            estado_parcial,
            "tiempo_transito_h",
        ] = (
            rutas_bloque.loc[
                estado_parcial,
                "tiempo_normal_h",
            ]
            * factor_tiempo
        )

        rutas_bloque.loc[
            estado_parcial,
            "capacidad_vehiculos_viaje",
        ] = np.floor(
            rutas_bloque.loc[
                estado_parcial,
                "capacidad_vehiculos_normal",
            ]
            * factor_capacidad
        ).astype(int)

    if rutas_actualizadas is not None:
        if rutas_actualizadas.empty:
            rutas_actualizadas = None

    if rutas_actualizadas is not None:
        columnas_actualizacion = {
            "ruta_id",
            "bloque",
            "ruta_disponible",
            "tiempo_transito_h",
            "capacidad_vehiculos_viaje",
        }

        faltantes_actualizacion = (
            columnas_actualizacion.difference(
                rutas_actualizadas.columns
            )
        )

        if faltantes_actualizacion:
            raise ValueError(
                "La actualización de rutas no contiene: "
                f"{sorted(faltantes_actualizacion)}"
            )

        actualizacion = rutas_actualizadas.copy()

        if actualizacion.duplicated(
            subset=["ruta_id", "bloque"]
        ).any():
            raise ValueError(
                "La actualización contiene combinaciones "
                "duplicadas de ruta_id y bloque."
            )

        bloques_validos = set(
            range(configuracion["n_pasos"])
        )

        if not set(actualizacion["bloque"]).issubset(
            bloques_validos
        ):
            raise ValueError(
                "La actualización contiene bloques fuera "
                "del horizonte de simulación."
            )

        actualizacion = actualizacion.rename(
            columns={
                "ruta_disponible": (
                    "ruta_disponible_actualizada"
                ),
                "tiempo_transito_h": (
                    "tiempo_transito_actualizado_h"
                ),
                "capacidad_vehiculos_viaje": (
                    "capacidad_actualizada"
                ),
            }
        )

        rutas_bloque = rutas_bloque.merge(
            actualizacion,
            on=["ruta_id", "bloque"],
            how="left",
            validate="one_to_one",
        )

        tiene_actualizacion = rutas_bloque[
            "ruta_disponible_actualizada"
        ].notna()

        rutas_bloque.loc[
            tiene_actualizacion,
            "ruta_disponible",
        ] = rutas_bloque.loc[
            tiene_actualizacion,
            "ruta_disponible_actualizada",
        ].astype(bool)

        rutas_bloque.loc[
            tiene_actualizacion,
            "tiempo_transito_h",
        ] = rutas_bloque.loc[
            tiene_actualizacion,
            "tiempo_transito_actualizado_h",
        ]

        rutas_bloque.loc[
            tiene_actualizacion,
            "capacidad_vehiculos_viaje",
        ] = rutas_bloque.loc[
            tiene_actualizacion,
            "capacidad_actualizada",
        ]

        rutas_bloque.loc[
            tiene_actualizacion,
            "fuente_estado_ruta",
        ] = "Output formal del Grupo 4"

    rutas_bloque.loc[
        ~rutas_bloque["ruta_disponible"],
        "capacidad_vehiculos_viaje",
    ] = 0

    rutas_bloque["capacidad_vehiculos_viaje"] = (
        rutas_bloque[
            "capacidad_vehiculos_viaje"
        ].fillna(0).astype(int)
    )

    columnas_salida = [
        "bloque",
        "tiempo_h",
        "ruta_id",
        "ruta_original",
        "origen_original",
        "centro",
        "destino",
        "zona",
        "distancia_km",
        "tiempo_normal_h",
        "estado_ruta_original",
        "ruta_disponible",
        "tiempo_transito_h",
        "capacidad_vehiculos_normal",
        "capacidad_vehiculos_viaje",
        "fuente_estado_ruta",
    ]

    return (
        rutas_bloque[columnas_salida]
        .sort_values(["bloque", "ruta_id"])
        .reset_index(drop=True)
    )

rutas_actualizadas_pendientes = pd.DataFrame()

df_rutas_simulacion = preparar_rutas_simulacion(
    rutas=datos_modelo["rutas"],
    configuracion=configuracion_modelo,
    rutas_actualizadas=rutas_actualizadas_pendientes,
)

datos_modelo["rutas_simulacion"] = df_rutas_simulacion

print("PREPARACIÓN DE RUTAS: PASA")
print("Filas esperadas:", 9 * N_PASOS)
print("Filas obtenidas:", len(df_rutas_simulacion))

display(
    df_rutas_simulacion[
        [
            "bloque",
            "ruta_id",
            "centro",
            "destino",
            "zona",
            "estado_ruta_original",
            "ruta_disponible",
            "tiempo_transito_h",
            "capacidad_vehiculos_viaje",
        ]
    ].head(9)
)

PREPARACIÓN DE RUTAS: PASA
Filas esperadas: 108
Filas obtenidas: 108


,bloque,ruta_id,centro,destino,zona,estado_ruta_original,ruta_disponible,tiempo_transito_h,capacidad_vehiculos_viaje
0,0,R01,Bodega Central CONRED,Albergue Estadio,NaN,Disponible,True,0.40,8
1,0,R02,Bodega Central CONRED,Albergue Z2,Z2,Disponible,True,0.90,6
2,0,R03,Bodega Central CONRED,Albergue Z5-1,Z5,Dañada — velocidad reducida 60%,True,7.00,4
3,0,R04,Bodega Central CONRED,Albergue Z5-2,Z5,Dañada — velocidad reducida 60%,True,8.75,3
4,0,R05,Almacén Norte,Albergue Z2,Z2,Disponible,True,0.20,10
5,0,R06,Almacén Norte,Albergue Estadio,NaN,Disponible,True,0.70,7
6,0,R07,Depósito Sur,Albergue Z3,Z3,Disponible,True,0.40,8
7,0,R08,Centro Logístico Z5,Albergue Z5-1,Z5,Parcialmente bloqueada,False,NaN,0
8,0,R09,Centro Logístico Z5,Albergue Z5-2,Z5,Parcialmente bloqueada,False,NaN,0


### Lógica de reposiciones

In [16]:
def preparar_reposiciones(
    reposicion: pd.DataFrame,
    centros: pd.DataFrame,
    configuracion: dict[str, Any],
    rng: np.random.Generator | None = None,
) -> pd.DataFrame:

    columnas_requeridas = {
        "bloque",
        "bloque_original",
        "hora_inicio",
        "hora_fin",
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
        "fuente",
    }

    faltantes = columnas_requeridas.difference(
        reposicion.columns
    )

    if faltantes:
        raise ValueError(
            "La tabla de reposiciones no contiene: "
            f"{sorted(faltantes)}"
        )

    centro_receptor = configuracion[
        "centro_receptor_reposicion"
    ]

    centros_validos = set(centros["centro"])

    if centro_receptor is None:
        raise ValueError(
            "No se ha definido un centro receptor para "
            "las reposiciones externas."
        )

    if centro_receptor not in centros_validos:
        raise ValueError(
            f"El centro receptor '{centro_receptor}' "
            "no existe en la tabla de centros."
        )

    bloques_validos = set(
        range(configuracion["n_pasos"])
    )

    if not set(reposicion["bloque"]).issubset(
        bloques_validos
    ):
        raise ValueError(
            "La tabla de reposiciones contiene bloques "
            "fuera del horizonte."
        )

    columnas_suministros = {
        "agua_adicional_litros": "agua",
        "alimentos_adicionales_raciones": "alimentos",
        "medicamentos_adicionales_kits": "medicamentos",
    }

    unidades = {
        "agua": "litros",
        "alimentos": "raciones",
        "medicamentos": "kits",
    }

    reposiciones_largas = []

    for columna_excel, suministro in (
        columnas_suministros.items()
    ):
        tabla_suministro = reposicion[
            [
                "bloque",
                "bloque_original",
                "hora_inicio",
                "hora_fin",
                "fuente",
                columna_excel,
            ]
        ].copy()

        tabla_suministro = tabla_suministro.rename(
            columns={
                columna_excel: "reposicion_programada",
            }
        )

        tabla_suministro["centro"] = centro_receptor
        tabla_suministro["suministro"] = suministro
        tabla_suministro["unidad"] = unidades[suministro]

        reposiciones_largas.append(
            tabla_suministro
        )

    reposiciones = pd.concat(
        reposiciones_largas,
        ignore_index=True,
    )

    reposiciones["reposicion_programada"] = pd.to_numeric(
        reposiciones["reposicion_programada"],
        errors="raise",
    )

    if (reposiciones["reposicion_programada"] < 0).any():
        raise ValueError(
            "Las reposiciones programadas no pueden ser negativas."
        )

    cv_reposicion = configuracion[
        "incertidumbre"
    ]["cv_reposicion"]

    if cv_reposicion < 0:
        raise ValueError(
            "cv_reposicion no puede ser negativo."
        )

    if cv_reposicion > 0:
        if rng is None:
            raise ValueError(
                "Se necesita rng cuando cv_reposicion "
                "es mayor que cero."
            )

        sigma_log = np.sqrt(
            np.log(1 + cv_reposicion**2)
        )
        media_log = -0.5 * sigma_log**2

        multiplicador = rng.lognormal(
            mean=media_log,
            sigma=sigma_log,
            size=len(reposiciones),
        )
    else:
        multiplicador = np.ones(
            len(reposiciones)
        )

    reposiciones["reposicion_recibida"] = (
        reposiciones["reposicion_programada"]
        * multiplicador
    )

    reposiciones["asignacion_provisional"] = True
    reposiciones["politica_asignacion"] = (
        "Centro receptor definido en configuracion_modelo"
    )

    columnas_salida = [
        "bloque",
        "bloque_original",
        "hora_inicio",
        "hora_fin",
        "centro",
        "suministro",
        "reposicion_programada",
        "reposicion_recibida",
        "unidad",
        "fuente",
        "asignacion_provisional",
        "politica_asignacion",
    ]

    return (
        reposiciones[columnas_salida]
        .sort_values(
            ["bloque", "centro", "suministro"]
        )
        .reset_index(drop=True)
    )


rng_prueba_reposicion = np.random.default_rng(
    SEMILLA_BASE
)

df_reposiciones_simulacion = preparar_reposiciones(
    reposicion=datos_modelo["reposicion"],
    centros=datos_modelo["centros"],
    configuracion=configuracion_modelo,
    rng=rng_prueba_reposicion,
)

datos_modelo[
    "reposiciones_simulacion"
] = df_reposiciones_simulacion

print("PREPARACIÓN DE REPOSICIONES: PASA")
print("Filas esperadas:", 6 * 3)
print(
    "Filas obtenidas:",
    len(df_reposiciones_simulacion),
)
print(
    "Centro receptor provisional:",
    df_reposiciones_simulacion[
        "centro"
    ].unique().tolist(),
)

display(
    df_reposiciones_simulacion[
        [
            "bloque",
            "centro",
            "suministro",
            "reposicion_programada",
            "reposicion_recibida",
            "fuente",
        ]
    ]
)

PREPARACIÓN DE REPOSICIONES: PASA
Filas esperadas: 18
Filas obtenidas: 18
Centro receptor provisional: ['Bodega Central CONRED']


,bloque,centro,suministro,reposicion_programada,reposicion_recibida,fuente
0,2,Bodega Central CONRED,agua,0,0.0,Sin reposición aún
1,2,Bodega Central CONRED,alimentos,0,0.0,Sin reposición aún
2,2,Bodega Central CONRED,medicamentos,0,0.0,Sin reposición aún
3,3,Bodega Central CONRED,agua,40000,40000.0,Cruz Roja — primer convoy
4,3,Bodega Central CONRED,alimentos,8000,8000.0,Cruz Roja — primer convoy
5,3,Bodega Central CONRED,medicamentos,200,200.0,Cruz Roja — primer convoy
6,4,Bodega Central CONRED,agua,80000,80000.0,Gobierno central
7,4,Bodega Central CONRED,alimentos,15000,15000.0,Gobierno central
8,4,Bodega Central CONRED,medicamentos,400,400.0,Gobierno central
9,6,Bodega Central CONRED,agua,120000,120000.0,Ayuda internacional


### Preparar inventarios inciales

In [17]:
def preparar_inventarios_iniciales(
    centros: pd.DataFrame,
) -> pd.DataFrame:

    columnas_requeridas = {
        "centro",
        "zona",
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
    }

    faltantes = columnas_requeridas.difference(
        centros.columns
    )

    if faltantes:
        raise ValueError(
            "La tabla de centros no contiene las columnas: "
            f"{sorted(faltantes)}"
        )

    columnas_inventario = {
        "agua_litros": "agua",
        "alimentos_raciones": "alimentos",
        "medicamentos_kits": "medicamentos",
    }

    unidades = {
        "agua": "litros",
        "alimentos": "raciones",
        "medicamentos": "kits",
    }

    inventarios_largos = []

    for columna_original, suministro in (
        columnas_inventario.items()
    ):
        tabla_suministro = centros[
            [
                "centro",
                "zona",
                columna_original,
            ]
        ].copy()

        tabla_suministro = tabla_suministro.rename(
            columns={
                columna_original: "inventario_inicial",
            }
        )

        tabla_suministro["suministro"] = suministro
        tabla_suministro["unidad"] = unidades[suministro]

        inventarios_largos.append(
            tabla_suministro
        )

    inventarios = pd.concat(
        inventarios_largos,
        ignore_index=True,
    )

    inventarios["inventario_inicial"] = pd.to_numeric(
        inventarios["inventario_inicial"],
        errors="raise",
    )

    if inventarios[
        ["centro", "zona", "suministro"]
    ].isna().any().any():
        raise ValueError(
            "Los inventarios iniciales contienen "
            "identificadores faltantes."
        )

    if (inventarios["inventario_inicial"] < 0).any():
        raise ValueError(
            "Los inventarios iniciales no pueden ser negativos."
        )

    if inventarios.duplicated(
        subset=["centro", "suministro"]
    ).any():
        raise ValueError(
            "Existen combinaciones duplicadas de "
            "centro y suministro."
        )

    columnas_salida = [
        "centro",
        "zona",
        "suministro",
        "inventario_inicial",
        "unidad",
    ]

    return (
        inventarios[columnas_salida]
        .sort_values(
            ["centro", "suministro"]
        )
        .reset_index(drop=True)
    )


df_inventarios_iniciales = (
    preparar_inventarios_iniciales(
        datos_modelo["centros"]
    )
)

datos_modelo[
    "inventarios_iniciales"
] = df_inventarios_iniciales

print("PREPARACIÓN DE INVENTARIOS: PASA")
print("Filas esperadas:", 4 * 3)
print(
    "Filas obtenidas:",
    len(df_inventarios_iniciales),
)

display(df_inventarios_iniciales)

PREPARACIÓN DE INVENTARIOS: PASA
Filas esperadas: 12
Filas obtenidas: 12


,centro,zona,suministro,inventario_inicial,unidad
0,Almacén Norte,Z2,agua,65000,litros
1,Almacén Norte,Z2,alimentos,18000,raciones
2,Almacén Norte,Z2,medicamentos,380,kits
3,Bodega Central CONRED,Z4,agua,180000,litros
4,Bodega Central CONRED,Z4,alimentos,42000,raciones
5,Bodega Central CONRED,Z4,medicamentos,1200,kits
6,Centro Logístico Z5,Z5,agua,28000,litros
7,Centro Logístico Z5,Z5,alimentos,7200,raciones
8,Centro Logístico Z5,Z5,medicamentos,145,kits
9,Depósito Sur,Z3,agua,40000,litros


### Función para simular el escenario

In [18]:
def simular_escenario(
    datos_modelo: dict[str, pd.DataFrame],
    configuracion: dict[str, Any],
    rng: np.random.Generator,
) -> dict[str, pd.DataFrame]:

    claves_requeridas = {
        "centros",
        "consumo",
        "rutas",
        "flota",
        "reposicion",
        "inventarios_iniciales",
        "demanda",
    }

    claves_faltantes = claves_requeridas.difference(
        datos_modelo.keys()
    )

    if claves_faltantes:
        raise KeyError(
            "Faltan estructuras requeridas para simular: "
            f"{sorted(claves_faltantes)}"
        )

    inventarios_iniciales = datos_modelo[
        "inventarios_iniciales"
    ].copy()

    demanda = datos_modelo["demanda"].copy()

    reposiciones = preparar_reposiciones(
        reposicion=datos_modelo["reposicion"],
        centros=datos_modelo["centros"],
        configuracion=configuracion,
        rng=rng,
    )

    if "rutas_simulacion" in datos_modelo:
        rutas_simulacion = datos_modelo[
            "rutas_simulacion"
        ].copy()
    else:
        rutas_simulacion = preparar_rutas_simulacion(
            rutas=datos_modelo["rutas"],
            configuracion=configuracion,
            rutas_actualizadas=None,
        )

    columnas_demanda = {
        "bloque",
        "zona",
        "suministro",
        "demanda",
    }

    if not demanda.empty:
        faltantes_demanda = columnas_demanda.difference(
            demanda.columns
        )

        if faltantes_demanda:
            raise ValueError(
                "La tabla de demanda no contiene: "
                f"{sorted(faltantes_demanda)}"
            )

        if demanda.duplicated(
            subset=["bloque", "zona", "suministro"]
        ).any():
            raise ValueError(
                "La demanda contiene combinaciones duplicadas "
                "de bloque, zona y suministro."
            )

        if (demanda["demanda"] < 0).any():
            raise ValueError(
                "La demanda no puede contener valores negativos."
            )

    estado_inicial = inventarios_iniciales[
        [
            "centro",
            "zona",
            "suministro",
            "inventario_inicial",
            "unidad",
        ]
    ].copy()

    zonas_con_centro = set(
        estado_inicial["zona"]
    )

    zonas_sin_centro = [
        zona
        for zona in configuracion["zonas_esperadas"]
        if zona not in zonas_con_centro
    ]

    registros_sin_centro = []

    unidades = {
        fila["suministro"]: fila["unidad_por_bloque"]
        .split("/persona/bloque")[0]
        for _, fila in datos_modelo["consumo"].iterrows()
    }

    unidades["agua"] = "litros"
    unidades["alimentos"] = "raciones"
    unidades["medicamentos"] = "kits"

    for zona in zonas_sin_centro:
        for suministro in configuracion[
            "suministros_esperados"
        ]:
            registros_sin_centro.append(
                {
                    "centro": "SIN_CENTRO_INICIAL",
                    "zona": zona,
                    "suministro": suministro,
                    "inventario_inicial": 0.0,
                    "unidad": unidades[suministro],
                }
            )

    if registros_sin_centro:
        estado_inicial = pd.concat(
            [
                estado_inicial,
                pd.DataFrame(registros_sin_centro),
            ],
            ignore_index=True,
        )

    if estado_inicial.duplicated(
        subset=["zona", "suministro"]
    ).any():
        raise ValueError(
            "Existe más de un centro asociado con la misma zona "
            "y suministro. Se necesita una política explícita "
            "para repartir la demanda entre centros."
        )

    estado_inventario = {
        (
            fila["centro"],
            fila["zona"],
            fila["suministro"],
        ): float(fila["inventario_inicial"])
        for _, fila in estado_inicial.iterrows()
    }

    if demanda.empty:
        demanda_por_clave = {}
    else:
        demanda_por_clave = (
            demanda.set_index(
                ["bloque", "zona", "suministro"]
            )["demanda"]
            .astype(float)
            .to_dict()
        )

    reposicion_por_clave = (
        reposiciones.groupby(
            ["bloque", "centro", "suministro"],
            as_index=True,
        )["reposicion_recibida"]
        .sum()
        .to_dict()
    )

    registros_balance = []

    for bloque in range(
        configuracion["n_pasos"]
    ):
        tiempo_h = (
            bloque * configuracion["paso_horas"]
        )

        actualizaciones_bloque = []

        for (
            centro,
            zona,
            suministro,
        ), inventario_inicio in estado_inventario.items():

            reposicion_recibida = float(
                reposicion_por_clave.get(
                    (bloque, centro, suministro),
                    0.0,
                )
            )

            demanda_bloque = float(
                demanda_por_clave.get(
                    (bloque, zona, suministro),
                    0.0,
                )
            )

            disponibilidad = (
                inventario_inicio
                + reposicion_recibida
            )

            consumo_satisfecho = min(
                disponibilidad,
                demanda_bloque,
            )

            demanda_no_satisfecha = max(
                0.0,
                demanda_bloque - disponibilidad,
            )

            inventario_fin = (
                disponibilidad
                - consumo_satisfecho
            )

            superavit = max(
                0.0,
                disponibilidad - demanda_bloque,
            )

            deficit = demanda_no_satisfecha

            if inventario_fin < -1e-9:
                raise RuntimeError(
                    "El inventario final no puede ser negativo."
                )

            actualizaciones_bloque.append(
                {
                    "clave": (
                        centro,
                        zona,
                        suministro,
                    ),
                    "inventario_fin": max(
                        0.0,
                        inventario_fin,
                    ),
                }
            )

            registros_balance.append(
                {
                    "escenario": configuracion.get(
                        "nombre_escenario",
                        "base_grupo3",
                    ),
                    "tiempo_h": tiempo_h,
                    "bloque": bloque,
                    "zona": zona,
                    "centro": centro,
                    "suministro": suministro,
                    "inventario_inicio": (
                        inventario_inicio
                    ),
                    "demanda": demanda_bloque,
                    "consumo_satisfecho": (
                        consumo_satisfecho
                    ),
                    "demanda_no_satisfecha": (
                        demanda_no_satisfecha
                    ),
                    "reposicion_recibida": (
                        reposicion_recibida
                    ),
                    "inventario_fin": inventario_fin,
                    "superavit": superavit,
                    "deficit": deficit,
                    "unidad": unidades[suministro],
                }
            )

        for actualizacion in actualizaciones_bloque:
            estado_inventario[
                actualizacion["clave"]
            ] = actualizacion[
                "inventario_fin"
            ]

    balance = pd.DataFrame(
        registros_balance
    ).sort_values(
        [
            "bloque",
            "zona",
            "centro",
            "suministro",
        ]
    ).reset_index(drop=True)

    columnas_logistica = [
        "escenario",
        "tiempo_h",
        "bloque",
        "ruta",
        "ruta_id",
        "centro",
        "destino",
        "zona",
        "vehiculo",
        "viajes",
        "carga_transportada",
        "combustible_consumido",
        "combustible_restante",
        "tiempo_transito",
    ]

    logistica = pd.DataFrame(
        columns=columnas_logistica
    )

    return {
        "balance": balance,
        "logistica": logistica,
        "rutas": rutas_simulacion,
        "reposiciones": reposiciones,
    }


rng_escenario_prueba = np.random.default_rng(
    SEMILLA_BASE
)

resultado_escenario = simular_escenario(
    datos_modelo=datos_modelo,
    configuracion=configuracion_modelo,
    rng=rng_escenario_prueba,
)

print("MOTOR GENERAL: PASA")
print(
    "Filas del balance:",
    len(resultado_escenario["balance"]),
)
print(
    "Filas logísticas:",
    len(resultado_escenario["logistica"]),
)
print(
    "Rutas por bloque:",
    len(resultado_escenario["rutas"]),
)

display(
    resultado_escenario["balance"].head(12)
)

MOTOR GENERAL: PASA
Filas del balance: 180
Filas logísticas: 0
Rutas por bloque: 108


,escenario,tiempo_h,bloque,zona,centro,suministro,inventario_inicio,demanda,consumo_satisfecho,demanda_no_satisfecha,reposicion_recibida,inventario_fin,superavit,deficit,unidad
0,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,agua,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,litros
1,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,alimentos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,raciones
2,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,medicamentos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,kits
3,base_grupo3,0,0,Z2,Almacén Norte,agua,65000.0,0.0,0.0,0.0,0.0,65000.0,65000.0,0.0,litros
4,base_grupo3,0,0,Z2,Almacén Norte,alimentos,18000.0,0.0,0.0,0.0,0.0,18000.0,18000.0,0.0,raciones
5,base_grupo3,0,0,Z2,Almacén Norte,medicamentos,380.0,0.0,0.0,0.0,0.0,380.0,380.0,0.0,kits
6,base_grupo3,0,0,Z3,Depósito Sur,agua,40000.0,0.0,0.0,0.0,0.0,40000.0,40000.0,0.0,litros
7,base_grupo3,0,0,Z3,Depósito Sur,alimentos,9500.0,0.0,0.0,0.0,0.0,9500.0,9500.0,0.0,raciones
8,base_grupo3,0,0,Z3,Depósito Sur,medicamentos,210.0,0.0,0.0,0.0,0.0,210.0,210.0,0.0,kits
9,base_grupo3,0,0,Z4,Bodega Central CONRED,agua,180000.0,0.0,0.0,0.0,0.0,180000.0,180000.0,0.0,litros


### Motor de realizaciones

In [19]:
def ejecutar_realizaciones(
    datos_modelo: dict[str, pd.DataFrame],
    configuracion: dict[str, Any],
    n_runs: int = N_RUNS,
    semilla_base: int = SEMILLA_BASE,
) -> pd.DataFrame:

    if n_runs < 30:
        raise ValueError(
            "El modelo debe ejecutar al menos 30 realizaciones."
        )

    if not isinstance(semilla_base, (int, np.integer)):
        raise TypeError(
            "semilla_base debe ser un número entero."
        )

    secuencia_base = np.random.SeedSequence(
        semilla_base
    )

    secuencias_realizaciones = (
        secuencia_base.spawn(n_runs)
    )

    resultados_balance = []

    for realizacion, secuencia in enumerate(
        secuencias_realizaciones
    ):
        rng = np.random.default_rng(secuencia)

        resultado = simular_escenario(
            datos_modelo=datos_modelo,
            configuracion=configuracion,
            rng=rng,
        )

        balance_realizacion = resultado[
            "balance"
        ].copy()

        semilla_realizacion = int(
            secuencia.generate_state(1)[0]
        )

        balance_realizacion.insert(
            0,
            "realizacion",
            realizacion,
        )

        balance_realizacion.insert(
            1,
            "semilla_realizacion",
            semilla_realizacion,
        )

        resultados_balance.append(
            balance_realizacion
        )

    resultados = pd.concat(
        resultados_balance,
        ignore_index=True,
    )

    columnas_orden = [
        "realizacion",
        "semilla_realizacion",
        "escenario",
        "tiempo_h",
        "bloque",
        "zona",
        "centro",
        "suministro",
        "inventario_inicio",
        "demanda",
        "consumo_satisfecho",
        "demanda_no_satisfecha",
        "reposicion_recibida",
        "inventario_fin",
        "superavit",
        "deficit",
        "unidad",
    ]

    resultados = (
        resultados[columnas_orden]
        .sort_values(
            [
                "realizacion",
                "bloque",
                "zona",
                "centro",
                "suministro",
            ]
        )
        .reset_index(drop=True)
    )

    return resultados


resultados_realizaciones = ejecutar_realizaciones(
    datos_modelo=datos_modelo,
    configuracion=configuracion_modelo,
    n_runs=N_RUNS,
    semilla_base=SEMILLA_BASE,
)

print("MOTOR DE REALIZACIONES: PASA")
print(
    "Realizaciones:",
    resultados_realizaciones[
        "realizacion"
    ].nunique(),
)
print(
    "Semillas diferentes:",
    resultados_realizaciones[
        [
            "realizacion",
            "semilla_realizacion",
        ]
    ].drop_duplicates()[
        "semilla_realizacion"
    ].nunique(),
)
print(
    "Filas totales:",
    len(resultados_realizaciones),
)

display(
    resultados_realizaciones.head(12)
)

MOTOR DE REALIZACIONES: PASA
Realizaciones: 100
Semillas diferentes: 100
Filas totales: 18000


,realizacion,semilla_realizacion,escenario,tiempo_h,bloque,zona,centro,suministro,inventario_inicio,demanda,consumo_satisfecho,demanda_no_satisfecha,reposicion_recibida,inventario_fin,superavit,deficit,unidad
0,0,479243620,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,agua,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,litros
1,0,479243620,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,alimentos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,raciones
2,0,479243620,base_grupo3,0,0,Z1,SIN_CENTRO_INICIAL,medicamentos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,kits
3,0,479243620,base_grupo3,0,0,Z2,Almacén Norte,agua,65000.0,0.0,0.0,0.0,0.0,65000.0,65000.0,0.0,litros
4,0,479243620,base_grupo3,0,0,Z2,Almacén Norte,alimentos,18000.0,0.0,0.0,0.0,0.0,18000.0,18000.0,0.0,raciones
5,0,479243620,base_grupo3,0,0,Z2,Almacén Norte,medicamentos,380.0,0.0,0.0,0.0,0.0,380.0,380.0,0.0,kits
6,0,479243620,base_grupo3,0,0,Z3,Depósito Sur,agua,40000.0,0.0,0.0,0.0,0.0,40000.0,40000.0,0.0,litros
7,0,479243620,base_grupo3,0,0,Z3,Depósito Sur,alimentos,9500.0,0.0,0.0,0.0,0.0,9500.0,9500.0,0.0,raciones
8,0,479243620,base_grupo3,0,0,Z3,Depósito Sur,medicamentos,210.0,0.0,0.0,0.0,0.0,210.0,210.0,0.0,kits
9,0,479243620,base_grupo3,0,0,Z4,Bodega Central CONRED,agua,180000.0,0.0,0.0,0.0,0.0,180000.0,180000.0,0.0,litros


## Pruebas de consistencia

In [20]:
def ejecutar_pruebas_consistencia(
    datos_modelo: dict[str, pd.DataFrame],
    configuracion: dict[str, Any],
) -> pd.DataFrame:

    resultados_pruebas = []

    def registrar_prueba(
        nombre: str,
        condicion: bool,
        evidencia: str,
    ) -> None:
        estado = "PASA" if condicion else "FALLA"

        resultados_pruebas.append(
            {
                "prueba": nombre,
                "resultado": estado,
                "evidencia": evidencia,
            }
        )

        if not condicion:
            raise AssertionError(
                f"{nombre}: {evidencia}"
            )

    registrar_prueba(
        nombre="Cantidad de centros",
        condicion=len(datos_modelo["centros"]) == 4,
        evidencia=(
            f"Se cargaron "
            f"{len(datos_modelo['centros'])} centros."
        ),
    )

    registrar_prueba(
        nombre="Cantidad de suministros",
        condicion=len(datos_modelo["consumo"]) == 3,
        evidencia=(
            f"Se cargaron "
            f"{len(datos_modelo['consumo'])} suministros."
        ),
    )

    registrar_prueba(
        nombre="Cantidad de rutas",
        condicion=len(datos_modelo["rutas"]) == 9,
        evidencia=(
            f"Se cargaron "
            f"{len(datos_modelo['rutas'])} rutas."
        ),
    )

    registrar_prueba(
        nombre="Tipos de vehículo",
        condicion=len(datos_modelo["flota"]) == 3,
        evidencia=(
            f"Se cargaron "
            f"{len(datos_modelo['flota'])} tipos de vehículo."
        ),
    )

    registrar_prueba(
        nombre="Cantidad de bloques",
        condicion=configuracion["n_pasos"] == 12,
        evidencia=(
            f"El horizonte contiene "
            f"{configuracion['n_pasos']} bloques."
        ),
    )

    consumo = datos_modelo["consumo"]

    tasas_calculadas = (
        consumo["consumo_por_persona_dia"]
        * configuracion["paso_horas"]
        / 24
    )

    conversion_correcta = np.allclose(
        consumo["consumo_por_persona_bloque"],
        tasas_calculadas,
    )

    registrar_prueba(
        nombre="Conversión de consumo",
        condicion=conversion_correcta,
        evidencia=(
            "Las tasas por bloque coinciden con "
            "consumo_diario × 6/24."
        ),
    )

    datos_sin_flujos = {
        clave: (
            valor.copy(deep=True)
            if isinstance(valor, pd.DataFrame)
            else valor
        )
        for clave, valor in datos_modelo.items()
    }

    datos_sin_flujos["demanda"] = pd.DataFrame(
        columns=[
            "bloque",
            "zona",
            "suministro",
            "demanda",
        ]
    )

    columnas_reposicion = [
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
    ]

    datos_sin_flujos["reposicion"].loc[
        :,
        columnas_reposicion,
    ] = 0

    resultado_sin_flujos = simular_escenario(
        datos_modelo=datos_sin_flujos,
        configuracion=configuracion,
        rng=np.random.default_rng(SEMILLA_BASE),
    )["balance"]

    inventario_constante = np.allclose(
        resultado_sin_flujos["inventario_inicio"],
        resultado_sin_flujos["inventario_fin"],
    )

    registrar_prueba(
        nombre="Conservación sin flujos",
        condicion=inventario_constante,
        evidencia=(
            "Sin demanda y sin reposición, el inventario "
            "inicial y final coinciden en todas las filas."
        ),
    )

    combinaciones_prueba = (
        pd.MultiIndex.from_product(
            [
                range(configuracion["n_pasos"]),
                configuracion["zonas_esperadas"],
            ],
            names=["bloque", "zona"],
        )
        .to_frame(index=False)
    )

    combinaciones_prueba["personas"] = 100

    demanda_prueba = crear_tabla_demanda(
        poblacion=combinaciones_prueba,
        consumo=datos_modelo["consumo"],
        configuracion=configuracion,
        rng=np.random.default_rng(SEMILLA_BASE),
    )

    datos_demanda_prueba = {
        clave: (
            valor.copy(deep=True)
            if isinstance(valor, pd.DataFrame)
            else valor
        )
        for clave, valor in datos_sin_flujos.items()
    }

    datos_demanda_prueba["demanda"] = demanda_prueba

    resultado_demanda_prueba = simular_escenario(
        datos_modelo=datos_demanda_prueba,
        configuracion=configuracion,
        rng=np.random.default_rng(SEMILLA_BASE),
    )["balance"]

    inventario_no_aumenta = (
        resultado_demanda_prueba["inventario_fin"]
        <=
        resultado_demanda_prueba["inventario_inicio"]
        + 1e-9
    ).all()

    registrar_prueba(
        nombre="Inventario con consumo",
        condicion=inventario_no_aumenta,
        evidencia=(
            "Con demanda positiva y sin reposición, "
            "ningún inventario aumentó."
        ),
    )

    inventarios_no_negativos = (
        resultado_demanda_prueba["inventario_fin"]
        >= -1e-9
    ).all()

    registrar_prueba(
        nombre="Inventarios no negativos",
        condicion=inventarios_no_negativos,
        evidencia=(
            "Todos los inventarios finales son mayores "
            "o iguales a cero."
        ),
    )

    deficit_no_negativo = (
        resultado_demanda_prueba[
            "demanda_no_satisfecha"
        ] >= -1e-9
    ).all()

    registrar_prueba(
        nombre="Demanda no satisfecha no negativa",
        condicion=deficit_no_negativo,
        evidencia=(
            "La demanda no satisfecha nunca toma "
            "valores negativos."
        ),
    )

    bloques_observados = set(
        resultado_demanda_prueba["bloque"]
    )

    registrar_prueba(
        nombre="Doce bloques sin paso adicional",
        condicion=(
            bloques_observados == set(range(12))
            and len(bloques_observados) == 12
        ),
        evidencia=(
            f"Bloques observados: "
            f"{sorted(bloques_observados)}."
        ),
    )

    tiempos_observados = sorted(
        resultado_demanda_prueba[
            "tiempo_h"
        ].unique().tolist()
    )

    tiempos_esperados = list(
        range(
            0,
            configuracion["horizonte_horas"],
            configuracion["paso_horas"],
        )
    )

    registrar_prueba(
        nombre="Intervalos de seis horas",
        condicion=(
            tiempos_observados == tiempos_esperados
        ),
        evidencia=(
            f"Tiempos observados: {tiempos_observados}."
        ),
    )

    resultado_semilla_1 = simular_escenario(
        datos_modelo=datos_modelo,
        configuracion=configuracion,
        rng=np.random.default_rng(SEMILLA_BASE),
    )["balance"]

    resultado_semilla_2 = simular_escenario(
        datos_modelo=datos_modelo,
        configuracion=configuracion,
        rng=np.random.default_rng(SEMILLA_BASE),
    )["balance"]

    try:
        pd.testing.assert_frame_equal(
            resultado_semilla_1,
            resultado_semilla_2,
            check_exact=True,
        )
        misma_semilla_reproduce = True
    except AssertionError:
        misma_semilla_reproduce = False

    registrar_prueba(
        nombre="Reproducibilidad por semilla",
        condicion=misma_semilla_reproduce,
        evidencia=(
            "Dos ejecuciones con la misma semilla "
            "producen la misma trayectoria."
        ),
    )

    return pd.DataFrame(resultados_pruebas)


reporte_pruebas = ejecutar_pruebas_consistencia(
    datos_modelo=datos_modelo,
    configuracion=configuracion_modelo,
)

print("PRUEBAS DE CONSISTENCIA COMPLETADAS")
display(reporte_pruebas)

PRUEBAS DE CONSISTENCIA COMPLETADAS


,prueba,resultado,evidencia
0,Cantidad de centros,PASA,Se cargaron 4 centros.
1,Cantidad de suministros,PASA,Se cargaron 3 suministros.
2,Cantidad de rutas,PASA,Se cargaron 9 rutas.
3,Tipos de vehículo,PASA,Se cargaron 3 tipos de vehículo.
4,Cantidad de bloques,PASA,El horizonte contiene 12 bloques.
5,Conversión de consumo,PASA,Las tasas por bloque coinciden con consumo_dia...
6,Conservación sin flujos,PASA,"Sin demanda y sin reposición, el inventario in..."
7,Inventario con consumo,PASA,"Con demanda positiva y sin reposición, ningún ..."
8,Inventarios no negativos,PASA,Todos los inventarios finales son mayores o ig...
9,Demanda no satisfecha no negativa,PASA,La demanda no satisfecha nunca toma valores ne...


## Configuración de incertidumbre

El modelo debe ejecutarse mediante al menos 30 realizaciones y reportar medias e intervalos de confianza del 95 %. Sin embargo, el Excel del Grupo 3 no proporciona distribuciones probabilísticas, desviaciones estándar ni rangos de variación para la demanda, los tiempos de viaje o las reposiciones.

Para evitar introducir variabilidad arbitraria, los parámetros estocásticos se centralizan en `configuracion_modelo["incertidumbre"]` y permanecen inicialmente en cero:

- `cv_demanda`: coeficiente de variación de la demanda.
- `cv_tiempo_ruta`: coeficiente de variación de los tiempos de tránsito.
- `cv_reposicion`: coeficiente de variación de las cantidades recibidas.

El coeficiente de variación de una variable aleatoria $X$ se define como:

$$
CV_X
=
\frac{\sigma_X}{\mu_X},
$$

donde $\sigma_X$ es la desviación estándar y $\mu_X$ es la media.

Cuando un coeficiente sea mayor que cero, se utilizará un multiplicador lognormal positivo con media igual a uno. Esta elección evita generar demandas, tiempos o reposiciones negativas. Si $M$ es el multiplicador, sus parámetros se calculan mediante:

$$
\sigma_{\log}
=
\sqrt{
\ln(1+CV^2)
},
$$

$$
\mu_{\log}
=
-\frac{1}{2}\sigma_{\log}^{2}.
$$

Entonces:

$$
M
\sim
\operatorname{Lognormal}
\left(
\mu_{\log},
\sigma_{\log}
\right),
$$

con:

$$
E[M]=1.
$$

El valor utilizado en una realización se obtiene como:

$$
X^{(j)}
=
X_{\text{base}}M^{(j)},
$$

donde $j$ identifica la realización. De esta forma, la incertidumbre modifica la dispersión sin cambiar sistemáticamente el valor medio del dato original.

En la fase actual se utiliza:

| Parámetro | Valor inicial | Justificación |
|---|---:|---|
| `cv_demanda` | 0.0 | La demanda formal del Grupo 1 aún no se ha recibido |
| `cv_tiempo_ruta` | 0.0 | No existe un rango de variabilidad en el Excel y falta el output del Grupo 4 |
| `cv_reposicion` | 0.0 | El Excel presenta cantidades programadas sin distribución o rango |

Por tanto, las 100 realizaciones iniciales son deterministas y se utilizan para comprobar la reproducibilidad y estructura del pipeline. No deben presentarse como evidencia de incertidumbre en los resultados finales.

Después del intercambio, los coeficientes solo se modificarán si los Grupos 1 o 4 proporcionan distribuciones, rangos o información que permita estimarlos. Si esa información no está disponible, cualquier escenario de sensibilidad adicional se identificará explícitamente como supuesto y se reportará por separado del escenario base.

Para cada combinación de bloque, zona y suministro, el intervalo de confianza del 95 % se calculará a partir de las realizaciones como:

$$
IC_{95\%}
=
\bar{x}
\pm
t_{0.975,n-1}
\frac{s}{\sqrt{n}},
$$

donde:

- $\bar{x}$ es la media de las realizaciones.
- $s$ es la desviación estándar muestral.
- $n$ es el número de realizaciones.
- $t_{0.975,n-1}$ es el valor crítico de la distribución $t$ de Student.

Si las realizaciones permanecen deterministas, entonces $s=0$ y los límites del intervalo coinciden con la media. Esto no representa incertidumbre nula en la realidad; únicamente indica que todavía no se ha parametrizado una fuente de variación respaldada por datos.

## Ejecución de realizaciones

In [21]:
def ejecutar_realizaciones(
    datos_modelo: dict[str, pd.DataFrame],
    configuracion: dict[str, Any],
    n_runs: int = N_RUNS,
    semilla_base: int = SEMILLA_BASE,
) -> pd.DataFrame:

    if n_runs < 30:
        raise ValueError(
            "El modelo debe ejecutar al menos 30 realizaciones."
        )

    if not isinstance(semilla_base, (int, np.integer)):
        raise TypeError(
            "semilla_base debe ser un número entero."
        )

    secuencia_base = np.random.SeedSequence(
        semilla_base
    )

    secuencias_realizaciones = (
        secuencia_base.spawn(n_runs)
    )

    resultados_balance = []

    for realizacion, secuencia in enumerate(
        secuencias_realizaciones
    ):

        semilla_realizacion = int(
            secuencia.generate_state(1)[0]
        )

        rng = np.random.default_rng(
            semilla_realizacion
        )

        resultado = simular_escenario(
            datos_modelo=datos_modelo,
            configuracion=configuracion,
            rng=rng,
        )

        balance_realizacion = resultado[
            "balance"
        ].copy()

        balance_realizacion.insert(
            0,
            "semilla_realizacion",
            semilla_realizacion,
        )

        balance_realizacion.insert(
            0,
            "realizacion",
            realizacion,
        )

        resultados_balance.append(
            balance_realizacion
        )

    resultados_combinados = pd.concat(
        resultados_balance,
        ignore_index=True,
    )

    return (
        resultados_combinados
        .sort_values(
            [
                "realizacion",
                "bloque",
                "zona",
                "centro",
                "suministro",
            ]
        )
        .reset_index(drop=True)
    )